# Inspect the current project

In [10]:

from pathlib import Path

# GitHub repository URL
REPO_URL = (
    "https://github.com/vipulkumawat17/"
    "Sales-analytics-data-pipeline"
)

# Local project directory in Google Drive
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

# Data directories
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "olist"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Python source directory
SRC_DIR = PROJECT_ROOT / "src"

# Verify the project root only
assert PROJECT_ROOT.is_dir(), (
    f"Project directory not found: {PROJECT_ROOT}"
)

print("Project root verified:", PROJECT_ROOT)
print("GitHub repository:", REPO_URL)


Project root verified: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline
GitHub repository: https://github.com/vipulkumawat17/Sales-analytics-data-pipeline


Inspect existing PostgreSQL schema

In [12]:
%pip install "psycopg[binary]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.5/215.5 kB 14.3 MB/s eta 0:00:00


In [14]:

from google.colab import userdata
import psycopg

database_url = userdata.get("NEON_DB_URL")

query = """
SELECT
    table_schema,
    table_name,
    constraint_type,
    constraint_name
FROM information_schema.table_constraints
WHERE table_schema = 'public'
ORDER BY table_name, constraint_type, constraint_name;
"""

with psycopg.connect(database_url) as conn:
    with conn.cursor() as cur:
        cur.execute(query)
        rows = cur.fetchall()

print("Existing PostgreSQL tables and constraints:")
for row in rows:
    print(
        f"Schema: {row[0]} | Table: {row[1]} | "
        f"Constraint type: {row[2]} | Name: {row[3]}"
    )


Existing PostgreSQL tables and constraints:
Schema: public | Table: category_analytics | Constraint type: CHECK | Name: category_analytics_product_category_name_english_not_null
Schema: public | Table: category_analytics | Constraint type: PRIMARY KEY | Name: category_analytics_pkey
Schema: public | Table: customer_analytics | Constraint type: CHECK | Name: customer_analytics_customer_unique_id_not_null
Schema: public | Table: customer_analytics | Constraint type: PRIMARY KEY | Name: customer_analytics_pkey
Schema: public | Table: monthly_sales_analytics | Constraint type: CHECK | Name: monthly_sales_analytics_order_month_not_null
Schema: public | Table: monthly_sales_analytics | Constraint type: PRIMARY KEY | Name: monthly_sales_analytics_pkey
Schema: public | Table: order_analytics | Constraint type: CHECK | Name: order_analytics_customer_id_not_null
Schema: public | Table: order_analytics | Constraint type: CHECK | Name: order_analytics_order_id_not_null
Schema: public | Table: orde

Inspect database.py

In [15]:

from pathlib import Path

database_file = (
    PROJECT_ROOT / "src" / "sales_pipeline" / "database.py"
)

if not database_file.is_file():
    raise FileNotFoundError(
        f"Database module not found: {database_file}"
    )

print(database_file.read_text(encoding="utf-8"))


"""Database loading functions for the sales analytics warehouse."""

import psycopg


CREATE_TABLES_SQL = """
CREATE TABLE IF NOT EXISTS order_analytics (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL,
    order_status TEXT,
    order_purchase_timestamp TIMESTAMP,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date DATE,
    item_count INTEGER,
    unique_products INTEGER,
    unique_sellers INTEGER,
    item_revenue NUMERIC(14, 2),
    freight_cost NUMERIC(14, 2),
    payment_total NUMERIC(14, 2),
    payment_records INTEGER,
    payment_types INTEGER,
    delivery_duration_days DOUBLE PRECISION,
    estimated_delivery_duration_days DOUBLE PRECISION,
    delivery_delay_days DOUBLE PRECISION,
    is_delivered BOOLEAN,
    is_late_delivery BOOLEAN
);

CREATE TABLE IF NOT EXISTS product_analytics (
    product_id TEXT PRIMARY KEY,
    product_category_name_english TEXT,


Inspect the existing pipeline test

In [16]:

test_file = PROJECT_ROOT / "tests" / "test_pipeline.py"

if not test_file.is_file():
    raise FileNotFoundError(f"Test file not found: {test_file}")

print(test_file.read_text(encoding="utf-8"))


"""Tests for sales analytics pipeline outputs."""

import unittest
from pathlib import Path
import pandas as pd


REPO_ROOT = Path(__file__).resolve().parents[1]
PROCESSED_DIR = REPO_ROOT / "data" / "processed"


class TestPipelineOutputs(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.orders = pd.read_csv(
            PROCESSED_DIR / "order_analytics.csv"
        )
        cls.products = pd.read_csv(
            PROCESSED_DIR / "product_analytics.csv"
        )
        cls.categories = pd.read_csv(
            PROCESSED_DIR / "category_analytics.csv"
        )
        cls.customers = pd.read_csv(
            PROCESSED_DIR / "customer_analytics.csv"
        )
        cls.monthly = pd.read_csv(
            PROCESSED_DIR / "monthly_sales_analytics.csv"
        )
        cls.status = pd.read_csv(
            PROCESSED_DIR / "order_status_summary.csv"
        )

    def test_order_ids_are_unique(self):
        self.assertEqual(
            self.orders["order_id"]

Inspect pipeline.py

In [17]:

pipeline_file = (
    PROJECT_ROOT / "src" / "sales_pipeline" / "pipeline.py"
)

if not pipeline_file.is_file():
    raise FileNotFoundError(f"Pipeline file not found: {pipeline_file}")

print(pipeline_file.read_text(encoding="utf-8"))


"""Orchestrate the sales analytics data pipeline."""

from pathlib import Path
from datetime import datetime, timezone

from . import validate
from . import transform
from . import transform_products
from . import transform_customers
from . import transform_sales
from . import database


def run_pipeline(data_dir, output_dir, database_url=None):
    data_dir = Path(data_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    started_at = datetime.now(timezone.utc)

    print("=" * 64)
    print("SALES ANALYTICS DATA PIPELINE")
    print("=" * 64)

    # Step 1: Validate source data.
    print("\n[1/5] Validating source data...")
    validation_results = validate.validate_data(data_dir)
    validation_passed = validate.print_report(validation_results)

    if not validation_passed:
        raise RuntimeError(
            "Pipeline stopped: one or more data-quality checks failed."
        )

    # Step 2: Order analytics.
    print("\n[2/5] Building 

In [18]:

from google.colab import userdata
import psycopg

database_url = userdata.get("NEON_DB_URL")

with psycopg.connect(database_url) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version(), current_database()")
        version, database_name = cur.fetchone()

print("PostgreSQL connection: successful")
print("Database:", database_name)
print("Server version:", version.split(",")[0])


PostgreSQL connection: successful
Database: neondb
Server version: PostgreSQL 18.6 (653b019) on aarch64-unknown-linux-gnu


# Design the batch-control table

### Batch metadata design

Create the batch control table

In [19]:

from google.colab import userdata
import psycopg

database_url = userdata.get("NEON_DB_URL")

CREATE_BATCH_TABLE_SQL = """
CREATE TABLE IF NOT EXISTS public.pipeline_batch_runs (
    batch_id TEXT PRIMARY KEY,
    batch_type TEXT NOT NULL
        CHECK (batch_type IN ('incremental', 'full_refresh')),
    status TEXT NOT NULL
        CHECK (status IN ('running', 'succeeded', 'failed')),
    started_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    finished_at TIMESTAMPTZ,
    source_rows BIGINT NOT NULL DEFAULT 0
        CHECK (source_rows >= 0),
    inserted_rows BIGINT NOT NULL DEFAULT 0
        CHECK (inserted_rows >= 0),
    updated_rows BIGINT NOT NULL DEFAULT 0
        CHECK (updated_rows >= 0),
    rejected_rows BIGINT NOT NULL DEFAULT 0
        CHECK (rejected_rows >= 0),
    error_message TEXT,
    attempt_number INTEGER NOT NULL DEFAULT 1
        CHECK (attempt_number >= 1)
);
"""

with psycopg.connect(database_url) as conn:
    with conn.cursor() as cur:
        cur.execute(CREATE_BATCH_TABLE_SQL)

print("Batch-control table is ready.")


Batch-control table is ready.


In [20]:

from google.colab import userdata
import psycopg

database_url = userdata.get("NEON_DB_URL")

with psycopg.connect(database_url) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                column_name,
                data_type,
                is_nullable,
                column_default
            FROM information_schema.columns
            WHERE table_schema = 'public'
              AND table_name = 'pipeline_batch_runs'
            ORDER BY ordinal_position;
        """)
        columns = cur.fetchall()

        cur.execute("""
            SELECT
                constraint_name,
                constraint_type
            FROM information_schema.table_constraints
            WHERE table_schema = 'public'
              AND table_name = 'pipeline_batch_runs'
            ORDER BY constraint_type, constraint_name;
        """)
        constraints = cur.fetchall()

print("Columns:")
for column in columns:
    print(column)

print("\nConstraints:")
for constraint in constraints:
    print(constraint)

assert columns, "Batch-control table was not found."
assert any(c[0] == "batch_id" for c in columns)
assert any(c[0] == "status" for c in columns)
assert any(c[0] == "attempt_number" for c in columns)

print("\nVerification passed.")


Columns:
('batch_id', 'text', 'NO', None)
('batch_type', 'text', 'NO', None)
('status', 'text', 'NO', None)
('started_at', 'timestamp with time zone', 'NO', 'now()')
('finished_at', 'timestamp with time zone', 'YES', None)
('source_rows', 'bigint', 'NO', '0')
('inserted_rows', 'bigint', 'NO', '0')
('updated_rows', 'bigint', 'NO', '0')
('rejected_rows', 'bigint', 'NO', '0')
('error_message', 'text', 'YES', None)
('attempt_number', 'integer', 'NO', '1')

Constraints:
('pipeline_batch_runs_attempt_number_check', 'CHECK')
('pipeline_batch_runs_attempt_number_not_null', 'CHECK')
('pipeline_batch_runs_batch_id_not_null', 'CHECK')
('pipeline_batch_runs_batch_type_check', 'CHECK')
('pipeline_batch_runs_batch_type_not_null', 'CHECK')
('pipeline_batch_runs_inserted_rows_check', 'CHECK')
('pipeline_batch_runs_inserted_rows_not_null', 'CHECK')
('pipeline_batch_runs_rejected_rows_check', 'CHECK')
('pipeline_batch_runs_rejected_rows_not_null', 'CHECK')
('pipeline_batch_runs_source_rows_check', 'CHEC

### Define bacth identity and idempotency

Define the batch identity

In [21]:

import hashlib
import json

def make_batch_id(source_name: str, batch_key: str) -> str:
    """Create a deterministic ID for a logical source batch."""
    if not source_name.strip():
        raise ValueError("source_name cannot be empty")
    if not batch_key.strip():
        raise ValueError("batch_key cannot be empty")

    identity = json.dumps(
        {
            "source": source_name.strip(),
            "batch_key": batch_key.strip(),
        },
        sort_keys=True,
        separators=(",", ":"),
    )

    digest = hashlib.sha256(identity.encode("utf-8")).hexdigest()
    return f"batch_{digest[:32]}"


# Controlled examples; no database writes
batch_a_first_run = make_batch_id("olist_orders", "test-batch-001")
batch_a_rerun = make_batch_id("olist_orders", "test-batch-001")
batch_b = make_batch_id("olist_orders", "test-batch-002")
batch_other_source = make_batch_id("olist_payments", "test-batch-001")

assert batch_a_first_run == batch_a_rerun
assert batch_a_first_run != batch_b
assert batch_a_first_run != batch_other_source

print("Batch A:", batch_a_first_run)
print("Batch A rerun:", batch_a_rerun)
print("Batch B:", batch_b)
print("Different source:", batch_other_source)
print("\nAll deterministic identity tests passed.")



Batch A: batch_e571c9326541848eeb1dcb3884ec1031
Batch A rerun: batch_e571c9326541848eeb1dcb3884ec1031
Batch B: batch_81f273800635ee77e3f418b046f76c4a
Different source: batch_b22f72f017aec385839a503aa1321a07

All deterministic identity tests passed.


Implement batch registration


In [22]:

from google.colab import userdata
import psycopg

database_url = userdata.get("NEON_DB_URL")


def register_batch(batch_id: str, batch_type: str = "incremental") -> str:
    """
    Register a logical batch.

    Returns:
        'started' for a new batch or retry.
        'skipped' if the batch already succeeded.

    Raises:
        RuntimeError if the batch is already running.
    """
    if batch_type not in {"incremental", "full_refresh"}:
        raise ValueError("Unsupported batch_type")

    with psycopg.connect(database_url) as conn:
        with conn.cursor() as cur:
            # Serialize registration attempts for the same batch ID.
            cur.execute(
                """
                INSERT INTO public.pipeline_batch_runs
                    (batch_id, batch_type, status, attempt_number)
                VALUES (%s, %s, 'running', 1)
                ON CONFLICT (batch_id) DO NOTHING
                RETURNING batch_id
                """,
                (batch_id, batch_type),
            )

            if cur.fetchone() is not None:
                return "started"

            cur.execute(
                """
                SELECT status, batch_type, attempt_number
                FROM public.pipeline_batch_runs
                WHERE batch_id = %s
                FOR UPDATE
                """,
                (batch_id,),
            )
            row = cur.fetchone()

            if row is None:
                raise RuntimeError("Batch registration state unavailable")

            status, existing_type, attempt_number = row

            if existing_type != batch_type:
                raise ValueError(
                    "Batch ID already exists with a different batch_type"
                )

            if status == "succeeded":
                return "skipped"

            if status == "running":
                raise RuntimeError(
                    "Batch is already running; concurrent execution rejected"
                )

            if status == "failed":
                cur.execute(
                    """
                    UPDATE public.pipeline_batch_runs
                    SET status = 'running',
                        started_at = NOW(),
                        finished_at = NULL,
                        error_message = NULL,
                        attempt_number = %s
                    WHERE batch_id = %s
                    """,
                    (attempt_number + 1, batch_id),
                )
                return "started"

            raise RuntimeError(f"Unexpected batch status: {status}")


# Use a dedicated test batch ID; do not use a real production batch ID.
test_batch_id = make_batch_id(
    "phase5_registration_test",
    "registration-check-001",
)

registration_result = register_batch(test_batch_id)

print("Registration result:", registration_result)
print("Test batch ID:", test_batch_id)


Registration result: started
Test batch ID: batch_c37d42228135ff68a97eff8889352e2b


Varify the registered batch

In [24]:

from google.colab import userdata
import psycopg

database_url = userdata.get("NEON_DB_URL")

with psycopg.connect(database_url) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT
                batch_id,
                batch_type,
                status,
                attempt_number,
                started_at,
                finished_at
            FROM public.pipeline_batch_runs
            WHERE batch_id = %s
            """,
            (test_batch_id,),
        )
        batch = cur.fetchone()

assert batch is not None, "Registered batch was not found."
assert batch[2] == "running", f"Unexpected status: {batch[2]}"
assert batch[3] == 1, f"Unexpected attempt number: {batch[3]}"

print("Batch state verified:")
print("Batch ID:", batch[0])
print("Type:", batch[1])
print("Status:", batch[2])
print("Attempt:", batch[3])
print("Started at:", batch[4])
print("Finished at:", batch[5])


Batch state verified:
Batch ID: batch_c37d42228135ff68a97eff8889352e2b
Type: incremental
Status: running
Attempt: 1
Started at: 2026-10-17 17:38:28.178517+00:00
Finished at: None


Implement batch success and failure transitions.

In [25]:

from google.colab import userdata
import psycopg


database_url = userdata.get("NEON_DB_URL")


def mark_batch_succeeded(
    batch_id: str,
    source_rows: int = 0,
    inserted_rows: int = 0,
    updated_rows: int = 0,
    rejected_rows: int = 0,
) -> None:
    """Mark a running batch as successful and record its metrics."""

    counts = {
        "source_rows": source_rows,
        "inserted_rows": inserted_rows,
        "updated_rows": updated_rows,
        "rejected_rows": rejected_rows,
    }

    if any(value < 0 for value in counts.values()):
        raise ValueError("Row counts cannot be negative")

    with psycopg.connect(database_url) as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                UPDATE public.pipeline_batch_runs
                SET status = 'succeeded',
                    finished_at = NOW(),
                    source_rows = %s,
                    inserted_rows = %s,
                    updated_rows = %s,
                    rejected_rows = %s,
                    error_message = NULL
                WHERE batch_id = %s
                  AND status = 'running'
                RETURNING batch_id
                """,
                (
                    source_rows,
                    inserted_rows,
                    updated_rows,
                    rejected_rows,
                    batch_id,
                ),
            )

            if cur.fetchone() is None:
                raise RuntimeError(
                    "Batch not found or is not in running state"
                )


def mark_batch_failed(batch_id: str, error_message: str) -> None:
    """Mark a running batch as failed without exposing credentials."""

    if not error_message or not error_message.strip():
        raise ValueError("An error message is required")

    with psycopg.connect(database_url) as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                UPDATE public.pipeline_batch_runs
                SET status = 'failed',
                    finished_at = NOW(),
                    error_message = %s
                WHERE batch_id = %s
                  AND status = 'running'
                RETURNING batch_id
                """,
                (error_message[:4000], batch_id),
            )

            if cur.fetchone() is None:
                raise RuntimeError(
                    "Batch not found or is not in running state"
                )


print("Batch lifecycle functions defined successfully.")


Batch lifecycle functions defined successfully.


Test success, failure, rerun and retry

In [26]:

# Test A: Successful batch and idempotent rerun
mark_batch_succeeded(
    test_batch_id,
    source_rows=10,
    inserted_rows=8,
    updated_rows=2,
    rejected_rows=0,
)

rerun_result = register_batch(test_batch_id)
assert rerun_result == "skipped", rerun_result

print("PASS: Successful batch rerun is skipped.")


# Test B: Failed batch, retry, and successful completion
failure_test_id = make_batch_id(
    "phase5_lifecycle_test",
    "failure-retry-check-001",
)

first_result = register_batch(failure_test_id)
assert first_result == "started", first_result

mark_batch_failed(
    failure_test_id,
    "Controlled test failure",
)

retry_result = register_batch(failure_test_id)
assert retry_result == "started", retry_result

mark_batch_succeeded(
    failure_test_id,
    source_rows=5,
    inserted_rows=5,
    updated_rows=0,
    rejected_rows=0,
)

# Verify both persisted records
with psycopg.connect(database_url) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT batch_id, status, attempt_number,
                   source_rows, inserted_rows, error_message,
                   finished_at
            FROM public.pipeline_batch_runs
            WHERE batch_id IN (%s, %s)
            ORDER BY batch_id
            """,
            (test_batch_id, failure_test_id),
        )
        results = cur.fetchall()

for row in results:
    print("\nBatch:", row[0])
    print("Status:", row[1])
    print("Attempts:", row[2])
    print("Source rows:", row[3])
    print("Inserted rows:", row[4])
    print("Error:", row[5])
    print("Finished at:", row[6])

assert len(results) == 2
assert all(row[1] == "succeeded" for row in results)
assert all(row[6] is not None for row in results)

retry_row = next(row for row in results if row[0] == failure_test_id)
assert retry_row[2] == 2
assert retry_row[5] is None

print("\nAll batch lifecycle tests passed.")


PASS: Successful batch rerun is skipped.

Batch: batch_13536d59e0f8afd3c3fbdabe796e7e83
Status: succeeded
Attempts: 2
Source rows: 5
Inserted rows: 5
Error: None
Finished at: 2026-10-17 17:42:26.056914+00:00

Batch: batch_c37d42228135ff68a97eff8889352e2b
Status: succeeded
Attempts: 1
Source rows: 10
Inserted rows: 8
Error: None
Finished at: 2026-10-17 17:42:16.638379+00:00

All batch lifecycle tests passed.


Create *incremental.py*

In [27]:

from pathlib import Path

module_path = PROJECT_ROOT / "src" / "sales_pipeline" / "incremental.py"
module_path.parent.mkdir(parents=True, exist_ok=True)

if module_path.exists():
    print("File already exists; preserving it:")
    print(module_path)
else:
    module_code = '''"""Batch tracking and lifecycle management for incremental processing."""

import hashlib
import json

import psycopg


BATCH_TABLE = "public.pipeline_batch_runs"


def make_batch_id(source_name: str, batch_key: str) -> str:
    """Return a deterministic ID for a logical source batch."""
    if not isinstance(source_name, str) or not source_name.strip():
        raise ValueError("source_name cannot be empty")
    if not isinstance(batch_key, str) or not batch_key.strip():
        raise ValueError("batch_key cannot be empty")

    identity = json.dumps(
        {
            "source": source_name.strip(),
            "batch_key": batch_key.strip(),
        },
        sort_keys=True,
        separators=(",", ":"),
    )
    digest = hashlib.sha256(identity.encode("utf-8")).hexdigest()
    return f"batch_{digest[:32]}"


def register_batch(
    database_url: str,
    batch_id: str,
    batch_type: str = "incremental",
) -> str:
    """Register a batch or retry a failed batch.

    Returns 'started' for a new or retried batch and 'skipped' for
    an already successful batch. Concurrent running batches are rejected.
    """
    if not isinstance(batch_id, str) or not batch_id.strip():
        raise ValueError("batch_id cannot be empty")
    if batch_type not in {"incremental", "full_refresh"}:
        raise ValueError("Unsupported batch_type")

    with psycopg.connect(database_url) as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                INSERT INTO public.pipeline_batch_runs
                    (batch_id, batch_type, status, attempt_number)
                VALUES (%s, %s, 'running', 1)
                ON CONFLICT (batch_id) DO NOTHING
                RETURNING batch_id
                """,
                (batch_id, batch_type),
            )

            if cur.fetchone() is not None:
                return "started"

            cur.execute(
                """
                SELECT status, batch_type, attempt_number
                FROM public.pipeline_batch_runs
                WHERE batch_id = %s
                FOR UPDATE
                """,
                (batch_id,),
            )
            row = cur.fetchone()

            if row is None:
                raise RuntimeError("Batch registration state unavailable")

            status, existing_type, attempt_number = row

            if existing_type != batch_type:
                raise ValueError(
                    "Batch ID already exists with a different batch_type"
                )

            if status == "succeeded":
                return "skipped"

            if status == "running":
                raise RuntimeError(
                    "Batch is already running; concurrent execution rejected"
                )

            if status == "failed":
                cur.execute(
                    """
                    UPDATE public.pipeline_batch_runs
                    SET status = 'running',
                        started_at = NOW(),
                        finished_at = NULL,
                        error_message = NULL,
                        attempt_number = %s
                    WHERE batch_id = %s
                    """,
                    (attempt_number + 1, batch_id),
                )
                return "started"

            raise RuntimeError(f"Unexpected batch status: {status}")


def mark_batch_succeeded(
    database_url: str,
    batch_id: str,
    source_rows: int = 0,
    inserted_rows: int = 0,
    updated_rows: int = 0,
    rejected_rows: int = 0,
) -> None:
    """Mark a running batch successful and record its row counts."""
    counts = (source_rows, inserted_rows, updated_rows, rejected_rows)
    if any(not isinstance(value, int) or value < 0 for value in counts):
        raise ValueError("Row counts must be non-negative integers")

    with psycopg.connect(database_url) as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                UPDATE public.pipeline_batch_runs
                SET status = 'succeeded',
                    finished_at = NOW(),
                    source_rows = %s,
                    inserted_rows = %s,
                    updated_rows = %s,
                    rejected_rows = %s,
                    error_message = NULL
                WHERE batch_id = %s
                  AND status = 'running'
                RETURNING batch_id
                """,
                (
                    source_rows,
                    inserted_rows,
                    updated_rows,
                    rejected_rows,
                    batch_id,
                ),
            )
            if cur.fetchone() is None:
                raise RuntimeError(
                    "Batch not found or is not in running state"
                )


def mark_batch_failed(
    database_url: str,
    batch_id: str,
    error_message: str,
) -> None:
    """Mark a running batch failed with a diagnostic message."""
    if not isinstance(error_message, str) or not error_message.strip():
        raise ValueError("An error message is required")

    with psycopg.connect(database_url) as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                UPDATE public.pipeline_batch_runs
                SET status = 'failed',
                    finished_at = NOW(),
                    error_message = %s
                WHERE batch_id = %s
                  AND status = 'running'
                RETURNING batch_id
                """,
                (error_message[:4000], batch_id),
            )
            if cur.fetchone() is None:
                raise RuntimeError(
                    "Batch not found or is not in running state"
                )
'''

    module_path.write_text(module_code, encoding="utf-8")
    print("Created:", module_path)

# Verify that the file can be parsed without executing it.
import ast

source = module_path.read_text(encoding="utf-8")
ast.parse(source)

print("Python syntax check passed.")
print("File size:", module_path.stat().st_size, "bytes")


Created: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/incremental.py
Python syntax check passed.
File size: 5774 bytes


Add unit *test_incremental.py*

In [28]:

from pathlib import Path
import ast

test_path = PROJECT_ROOT / "tests" / "test_incremental.py"
test_path.parent.mkdir(parents=True, exist_ok=True)

if test_path.exists():
    print("Test file already exists; preserving it:")
    print(test_path)
else:
    test_code = '''"""Unit tests for incremental batch identity and validation."""

import sys
import unittest
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sales_pipeline.incremental import make_batch_id


class TestBatchIdentity(unittest.TestCase):

    def test_same_identity_produces_same_id(self):
        first = make_batch_id("olist_orders", "batch-001")
        second = make_batch_id("olist_orders", "batch-001")
        self.assertEqual(first, second)

    def test_different_batch_keys_produce_different_ids(self):
        first = make_batch_id("olist_orders", "batch-001")
        second = make_batch_id("olist_orders", "batch-002")
        self.assertNotEqual(first, second)

    def test_different_sources_produce_different_ids(self):
        first = make_batch_id("olist_orders", "batch-001")
        second = make_batch_id("olist_payments", "batch-001")
        self.assertNotEqual(first, second)

    def test_batch_id_has_expected_prefix_and_length(self):
        batch_id = make_batch_id("olist_orders", "batch-001")
        self.assertTrue(batch_id.startswith("batch_"))
        self.assertEqual(len(batch_id), 38)

    def test_empty_source_is_rejected(self):
        with self.assertRaises(ValueError):
            make_batch_id("", "batch-001")

    def test_empty_batch_key_is_rejected(self):
        with self.assertRaises(ValueError):
            make_batch_id("olist_orders", " ")

    def test_whitespace_is_normalized(self):
        first = make_batch_id(" olist_orders ", " batch-001 ")
        second = make_batch_id("olist_orders", "batch-001")
        self.assertEqual(first, second)


if __name__ == "__main__":
    unittest.main()
'''

    test_path.write_text(test_code, encoding="utf-8")
    print("Created:", test_path)

ast.parse(test_path.read_text(encoding="utf-8"))
print("Test file syntax check passed.")



Created: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/tests/test_incremental.py
Test file syntax check passed.


In [29]:

import subprocess
import sys

test_file = PROJECT_ROOT / "tests" / "test_incremental.py"

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        str(PROJECT_ROOT / "tests"),
        "-p",
        "test_incremental.py",
        "-v",
    ],
    cwd=str(PROJECT_ROOT),
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"Incremental unit tests failed with exit code {result.returncode}"
    )

print("Incremental unit tests passed.")



test_batch_id_has_expected_prefix_and_length (test_incremental.TestBatchIdentity.test_batch_id_has_expected_prefix_and_length) ... ok
test_different_batch_keys_produce_different_ids (test_incremental.TestBatchIdentity.test_different_batch_keys_produce_different_ids) ... ok
test_different_sources_produce_different_ids (test_incremental.TestBatchIdentity.test_different_sources_produce_different_ids) ... ok
test_empty_batch_key_is_rejected (test_incremental.TestBatchIdentity.test_empty_batch_key_is_rejected) ... ok
test_empty_source_is_rejected (test_incremental.TestBatchIdentity.test_empty_source_is_rejected) ... ok
test_same_identity_produces_same_id (test_incremental.TestBatchIdentity.test_same_identity_produces_same_id) ... ok
test_whitespace_is_normalized (test_incremental.TestBatchIdentity.test_whitespace_is_normalized) ... ok

----------------------------------------------------------------------
Ran 7 tests in 0.001s

OK

Incremental unit tests passed.


Database Integretion test

In [30]:

import sys
from pathlib import Path
from google.colab import userdata
import psycopg

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sales_pipeline.incremental import (
    make_batch_id,
    register_batch,
    mark_batch_succeeded,
)

database_url = userdata.get("NEON_DB_URL")

# Dedicated integration-test batch
batch_id = make_batch_id(
    "phase5_integration_test",
    "registration-success-rerun-001",
)

# Register the batch; a prior successful run should be skipped.
first_result = register_batch(
    database_url,
    batch_id,
    "incremental",
)

if first_result == "started":
    mark_batch_succeeded(
        database_url,
        batch_id,
        source_rows=3,
        inserted_rows=3,
        updated_rows=0,
        rejected_rows=0,
    )

second_result = register_batch(
    database_url,
    batch_id,
    "incremental",
)

with psycopg.connect(database_url) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT COUNT(*), MIN(status), MIN(attempt_number),
                   MIN(source_rows), MIN(inserted_rows)
            FROM public.pipeline_batch_runs
            WHERE batch_id = %s
            """,
            (batch_id,),
        )
        stored = cur.fetchone()

print("First registration:", first_result)
print("Rerun result:", second_result)
print("Stored state:", stored)

assert first_result in {"started", "skipped"}
assert second_result == "skipped"
assert stored == (1, "succeeded", 1, 3, 3)

print("Database integration test passed.")


First registration: started
Rerun result: skipped
Stored state: (1, 'succeeded', 1, 3, 3)
Database integration test passed.


In [31]:

from google.colab import userdata
import psycopg

from sales_pipeline.incremental import (
    make_batch_id,
    register_batch,
    mark_batch_failed,
    mark_batch_succeeded,
)

database_url = userdata.get("NEON_DB_URL")

batch_id = make_batch_id(
    "phase5_integration_test",
    "failure-retry-001",
)

# Register the test batch if it has not been registered.
first_result = register_batch(database_url, batch_id)

if first_result == "started":
    mark_batch_failed(
        database_url,
        batch_id,
        "Controlled integration-test failure",
    )

# Retry the failed batch.
retry_result = register_batch(database_url, batch_id)
assert retry_result == "started", retry_result

# Complete the retry successfully.
mark_batch_succeeded(
    database_url,
    batch_id,
    source_rows=5,
    inserted_rows=4,
    updated_rows=1,
    rejected_rows=0,
)

# Verify persisted state.
with psycopg.connect(database_url) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT status, attempt_number, source_rows,
                   inserted_rows, updated_rows, error_message
            FROM public.pipeline_batch_runs
            WHERE batch_id = %s
            """,
            (batch_id,),
        )
        result = cur.fetchone()

print("Initial registration:", first_result)
print("Retry registration:", retry_result)
print("Final stored state:", result)

assert result == ("succeeded", 2, 5, 4, 1, None)

print("Failure/retry integration test passed.")


Initial registration: started
Retry registration: started
Final stored state: ('succeeded', 2, 5, 4, 1, None)
Failure/retry integration test passed.


Create the integration test file

In [32]:

from pathlib import Path
import ast

test_path = (
    PROJECT_ROOT / "tests" / "test_incremental_integration.py"
)

if test_path.exists():
    print("File already exists; preserving it:")
    print(test_path)
else:
    test_code = '''"""Integration tests for incremental batch lifecycle.

Requires NEON_DB_URL to be configured in the environment.
Tests use uniquely identified records in pipeline_batch_runs.
"""

import os
import sys
import unittest
import uuid
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sales_pipeline.incremental import (
    make_batch_id,
    register_batch,
    mark_batch_failed,
    mark_batch_succeeded,
)


class TestIncrementalBatchIntegration(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.database_url = os.environ.get("NEON_DB_URL")
        if not cls.database_url:
            raise unittest.SkipTest(
                "Set NEON_DB_URL to run database integration tests."
            )

    def new_batch_id(self, label):
        unique_key = f"{label}-{uuid.uuid4().hex}"
        return make_batch_id("integration_test", unique_key)

    def test_successful_batch_rerun_is_skipped(self):
        batch_id = self.new_batch_id("success-rerun")

        self.assertEqual(
            register_batch(self.database_url, batch_id),
            "started",
        )

        mark_batch_succeeded(
            self.database_url,
            batch_id,
            source_rows=3,
            inserted_rows=3,
        )

        self.assertEqual(
            register_batch(self.database_url, batch_id),
            "skipped",
        )

    def test_failed_batch_can_be_retried(self):
        batch_id = self.new_batch_id("failure-retry")

        self.assertEqual(
            register_batch(self.database_url, batch_id),
            "started",
        )

        mark_batch_failed(
            self.database_url,
            batch_id,
            "Controlled integration-test failure",
        )

        self.assertEqual(
            register_batch(self.database_url, batch_id),
            "started",
        )

        mark_batch_succeeded(
            self.database_url,
            batch_id,
            source_rows=5,
            inserted_rows=4,
            updated_rows=1,
        )

        import psycopg
        with psycopg.connect(self.database_url) as conn:
            with conn.cursor() as cur:
                cur.execute(
                    """
                    SELECT status, attempt_number, source_rows,
                           inserted_rows, updated_rows, error_message
                    FROM public.pipeline_batch_runs
                    WHERE batch_id = %s
                    """,
                    (batch_id,),
                )
                result = cur.fetchone()

        self.assertEqual(
            result,
            ("succeeded", 2, 5, 4, 1, None),
        )


if __name__ == "__main__":
    unittest.main()
'''

    test_path.write_text(test_code, encoding="utf-8")
    print("Created:", test_path)

ast.parse(test_path.read_text(encoding="utf-8"))
print("Syntax check passed.")


Created: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/tests/test_incremental_integration.py
Syntax check passed.


In [33]:

import os
import subprocess
import sys
from google.colab import userdata

database_url = userdata.get("NEON_DB_URL")

test_file = (
    PROJECT_ROOT / "tests" / "test_incremental_integration.py"
)

test_env = os.environ.copy()
test_env["NEON_DB_URL"] = database_url

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        str(PROJECT_ROOT / "tests"),
        "-p",
        "test_incremental_integration.py",
        "-v",
    ],
    cwd=str(PROJECT_ROOT),
    env=test_env,
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"Integration tests failed with exit code {result.returncode}"
    )

print("All incremental integration tests passed.")



test_failed_batch_can_be_retried (test_incremental_integration.TestIncrementalBatchIntegration.test_failed_batch_can_be_retried) ... ok
test_successful_batch_rerun_is_skipped (test_incremental_integration.TestIncrementalBatchIntegration.test_successful_batch_rerun_is_skipped) ... ok

----------------------------------------------------------------------
Ran 2 tests in 14.726s

OK

All incremental integration tests passed.


# Implement actual incremental data writes

Create the Isolated incremental target table

In [35]:

from google.colab import userdata
import psycopg

database_url = userdata.get("NEON_DB_URL")

CREATE_INCREMENTAL_TARGET_SQL = """
CREATE TABLE IF NOT EXISTS public.phase5_incremental_orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL,
    order_status TEXT,
    order_purchase_timestamp TIMESTAMP,
    source_batch_id TEXT NOT NULL,
    updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
"""

with psycopg.connect(database_url) as conn:
    with conn.cursor() as cur:
        cur.execute(CREATE_INCREMENTAL_TARGET_SQL)

        cur.execute("""
            SELECT column_name, data_type, is_nullable
            FROM information_schema.columns
            WHERE table_schema = 'public'
              AND table_name = 'phase5_incremental_orders'
            ORDER BY ordinal_position
        """)
        columns = cur.fetchall()

print("Incremental target table verified:")
for column in columns:
    print(column)

assert [column[0] for column in columns] == [
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "source_batch_id",
    "updated_at",
]

assert columns[0][2] == "NO"




Incremental target table verified:
('order_id', 'text', 'NO')
('customer_id', 'text', 'NO')
('order_status', 'text', 'YES')
('order_purchase_timestamp', 'timestamp without time zone', 'YES')
('source_batch_id', 'text', 'NO')
('updated_at', 'timestamp with time zone', 'NO')


Implement the transactional incremental writer

In [36]:

from pathlib import Path
import ast

module_path = PROJECT_ROOT / "src" / "sales_pipeline" / "incremental.py"

existing_code = module_path.read_text(encoding="utf-8")

if "def write_incremental_orders(" in existing_code:
    print("write_incremental_orders already exists; no changes made.")
else:
    writer_code = '''

def write_incremental_orders(
    database_url: str,
    batch_id: str,
    records: list[dict],
) -> dict[str, int]:
    """Atomically upsert order records and mark their batch successful.

    Each record must contain:
      - order_id
      - customer_id
      - order_status
      - order_purchase_timestamp

    Target writes and successful batch metadata are committed together.
    """
    required_fields = {
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
    }

    if not isinstance(records, list):
        raise TypeError("records must be a list of dictionaries")

    seen_ids = set()
    for index, record in enumerate(records):
        if not isinstance(record, dict):
            raise TypeError(f"Record {index} must be a dictionary")

        missing = required_fields - record.keys()
        if missing:
            raise ValueError(
                f"Record {index} is missing fields: {sorted(missing)}"
            )

        order_id = record["order_id"]
        customer_id = record["customer_id"]

        if not isinstance(order_id, str) or not order_id.strip():
            raise ValueError(f"Record {index} has an invalid order_id")
        if not isinstance(customer_id, str) or not customer_id.strip():
            raise ValueError(f"Record {index} has an invalid customer_id")

        if order_id in seen_ids:
            raise ValueError(
                f"Duplicate order_id within input batch: {order_id}"
            )
        seen_ids.add(order_id)

    inserted_rows = 0
    updated_rows = 0

    with psycopg.connect(database_url) as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                SELECT status, batch_type
                FROM public.pipeline_batch_runs
                WHERE batch_id = %s
                FOR UPDATE
                """,
                (batch_id,),
            )
            batch = cur.fetchone()

            if batch is None:
                raise RuntimeError("Batch is not registered")

            if batch[0] != "running":
                raise RuntimeError(
                    f"Batch must be running, found status: {batch[0]}"
                )

            if batch[1] != "incremental":
                raise RuntimeError(
                    "Incremental writer requires an incremental batch"
                )

            for record in records:
                cur.execute(
                    """
                    INSERT INTO public.phase5_incremental_orders (
                        order_id,
                        customer_id,
                        order_status,
                        order_purchase_timestamp,
                        source_batch_id,
                        updated_at
                    )
                    VALUES (%s, %s, %s, %s, %s, NOW())
                    ON CONFLICT (order_id) DO UPDATE SET
                        customer_id = EXCLUDED.customer_id,
                        order_status = EXCLUDED.order_status,
                        order_purchase_timestamp =
                            EXCLUDED.order_purchase_timestamp,
                        source_batch_id = EXCLUDED.source_batch_id,
                        updated_at = NOW()
                    RETURNING (xmax = 0) AS was_inserted
                    """,
                    (
                        record["order_id"],
                        record["customer_id"],
                        record["order_status"],
                        record["order_purchase_timestamp"],
                        batch_id,
                    ),
                )

                was_inserted = cur.fetchone()[0]
                if was_inserted:
                    inserted_rows += 1
                else:
                    updated_rows += 1

            cur.execute(
                """
                UPDATE public.pipeline_batch_runs
                SET status = 'succeeded',
                    finished_at = NOW(),
                    source_rows = %s,
                    inserted_rows = %s,
                    updated_rows = %s,
                    rejected_rows = 0,
                    error_message = NULL
                WHERE batch_id = %s
                  AND status = 'running'
                RETURNING batch_id
                """,
                (
                    len(records),
                    inserted_rows,
                    updated_rows,
                    batch_id,
                ),
            )

            if cur.fetchone() is None:
                raise RuntimeError(
                    "Could not mark batch successful; transaction aborted"
                )

    return {
        "source_rows": len(records),
        "inserted_rows": inserted_rows,
        "updated_rows": updated_rows,
        "rejected_rows": 0,
    }
'''

    module_path.write_text(
        existing_code.rstrip() + writer_code + "\n",
        encoding="utf-8",
    )
    print("Added write_incremental_orders to:", module_path)

ast.parse(module_path.read_text(encoding="utf-8"))
print("Python syntax check passed.")


Added write_incremental_orders to: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/incremental.py
Python syntax check passed.


In [37]:

import sys
from pathlib import Path

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sales_pipeline.incremental import write_incremental_orders

valid_record = {
    "order_id": "phase5-test-order-001",
    "customer_id": "phase5-test-customer-001",
    "order_status": "delivered",
    "order_purchase_timestamp": "2018-01-01 10:00:00",
}

# Test 1: Missing required field
invalid_missing_field = {
    "order_id": "phase5-test-order-002",
    "customer_id": "phase5-test-customer-002",
    "order_status": "delivered",
}

try:
    write_incremental_orders(
        "unused",
        "unused",
        [invalid_missing_field],
    )
    raise AssertionError("Missing field was not rejected")
except ValueError as exc:
    print("PASS: Missing required field rejected:", exc)

# Test 2: Duplicate business key within one batch
try:
    write_incremental_orders(
        "unused",
        "unused",
        [valid_record, valid_record.copy()],
    )
    raise AssertionError("Duplicate order_id was not rejected")
except ValueError as exc:
    print("PASS: Duplicate order_id rejected:", exc)

# Test 3: Invalid record type
try:
    write_incremental_orders(
        "unused",
        "unused",
        ["not-a-record"],
    )
    raise AssertionError("Invalid record type was not rejected")
except TypeError as exc:
    print("PASS: Invalid record type rejected:", exc)

print("\nAll input validation tests passed.")


ImportError: cannot import name 'write_incremental_orders' from 'sales_pipeline.incremental' (/content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/incremental.py)

In [39]:
from datetime import datetime
from sales_pipeline.incremental import (
    make_batch_id,
    register_batch,
    write_incremental_orders,
)
from google.colab import userdata

database_url = userdata.get("NEON_DB_URL")

batch_id = make_batch_id(
    "phase5_writer_test",
    "insert-update-test-001",
)

# Register the batch. Stop if this batch was already completed.
registration = register_batch(
    database_url,
    batch_id,
    batch_type="incremental",
)
print("Batch registration:", registration)

if registration == "started":
    records = [
        {
            "order_id": "phase5_test_order_001",
            "customer_id": "phase5_test_customer_001",
            "order_status": "delivered",
            "order_purchase_timestamp": datetime(2026, 10, 17, 10, 0, 0),
        },
        {
            "order_id": "phase5_test_order_002",
            "customer_id": "phase5_test_customer_002",
            "order_status": "processing",
            "order_purchase_timestamp": datetime(2026, 10, 17, 11, 0, 0),
        },
    ]

    result = write_incremental_orders(
        database_url,
        batch_id,
        records,
    )
    print("Writer result:", result)
else:
    print("This batch was already handled; no write was attempted.")

Batch registration: started
Writer result: {'source_rows': 2, 'inserted_rows': 2, 'updated_rows': 0, 'rejected_rows': 0}


In [40]:
from datetime import datetime
import psycopg
from google.colab import userdata
from sales_pipeline.incremental import (
    make_batch_id,
    register_batch,
    write_incremental_orders,
)

database_url = userdata.get("NEON_DB_URL")

# 1. Verify that the already completed batch is skipped.
original_batch_id = make_batch_id(
    "phase5_writer_test",
    "insert-update-test-001",
)

rerun_status = register_batch(
    database_url,
    original_batch_id,
    batch_type="incremental",
)
print("Completed batch rerun:", rerun_status)

# 2. Start a new batch containing one existing order with changed values.
update_batch_id = make_batch_id(
    "phase5_writer_test",
    "update-existing-order-001",
)

update_status = register_batch(
    database_url,
    update_batch_id,
    batch_type="incremental",
)
print("Update batch registration:", update_status)

if update_status == "started":
    update_result = write_incremental_orders(
        database_url,
        update_batch_id,
        [
            {
                "order_id": "phase5_test_order_001",
                "customer_id": "phase5_test_customer_001",
                "order_status": "shipped",
                "order_purchase_timestamp": datetime(
                    2026, 10, 17, 10, 0, 0
                ),
            }
        ],
    )
    print("Update result:", update_result)

# 3. Verify the target contains one row per order ID and the updated status.
with psycopg.connect(database_url) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT order_id, order_status, source_batch_id
            FROM public.phase5_incremental_orders
            WHERE order_id IN (
                'phase5_test_order_001',
                'phase5_test_order_002'
            )
            ORDER BY order_id
            """
        )
        rows = cur.fetchall()

        cur.execute(
            """
            SELECT COUNT(*), COUNT(DISTINCT order_id)
            FROM public.phase5_incremental_orders
            WHERE order_id IN (
                'phase5_test_order_001',
                'phase5_test_order_002'
            )
            """
        )
        total_rows, distinct_order_ids = cur.fetchone()

print("Verified records:", rows)
print("Total rows:", total_rows)
print("Distinct order IDs:", distinct_order_ids)

assert rerun_status == "skipped"
assert update_status == "started"
assert total_rows == 2
assert distinct_order_ids == 2
assert any(
    row[0] == "phase5_test_order_001" and row[1] == "shipped"
    for row in rows
)

print("PASS: completed reruns are skipped and existing orders are updated.")

Completed batch rerun: skipped
Update batch registration: started
Update result: {'source_rows': 1, 'inserted_rows': 0, 'updated_rows': 1, 'rejected_rows': 0}
Verified records: [('phase5_test_order_001', 'shipped', 'batch_ab28bb6982a4851fe5a27d08f56b23a4'), ('phase5_test_order_002', 'processing', 'batch_e71b49336e35bab784929acf5eead4b6')]
Total rows: 2
Distinct order IDs: 2
PASS: completed reruns are skipped and existing orders are updated.


# Failure Handling and Recovery

Update the incremental writer

In [41]:
import ast
from pathlib import Path

module_path = (
    PROJECT_ROOT
    / "src"
    / "sales_pipeline"
    / "incremental.py"
)

source = module_path.read_text(encoding="utf-8")
tree = ast.parse(source)

writer_node = next(
    (
        node
        for node in tree.body
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
        and node.name == "write_incremental_orders"
    ),
    None,
)

if writer_node is None:
    raise RuntimeError(
        "write_incremental_orders() was not found. "
        "No file changes were made."
    )

replacement = '''def write_incremental_orders(
    database_url: str,
    batch_id: str,
    records: list[dict],
) -> dict[str, int]:
    """Atomically upsert orders and record batch success.

    If validation or database processing fails, attempt to mark
    the registered batch as failed after the write transaction rolls back.
    """
    try:
        required_fields = {
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
        }

        if not isinstance(records, list):
            raise TypeError("records must be a list of dictionaries")

        seen_ids = set()

        for index, record in enumerate(records):
            if not isinstance(record, dict):
                raise TypeError(
                    f"Record {index} must be a dictionary"
                )

            missing = required_fields - record.keys()
            if missing:
                raise ValueError(
                    f"Record {index} is missing fields: "
                    f"{sorted(missing)}"
                )

            order_id = record["order_id"]
            customer_id = record["customer_id"]

            if not isinstance(order_id, str) or not order_id.strip():
                raise ValueError(
                    f"Record {index} has an invalid order_id"
                )

            if (
                not isinstance(customer_id, str)
                or not customer_id.strip()
            ):
                raise ValueError(
                    f"Record {index} has an invalid customer_id"
                )

            if order_id in seen_ids:
                raise ValueError(
                    f"Duplicate order_id within input batch: {order_id}"
                )

            seen_ids.add(order_id)

        inserted_rows = 0
        updated_rows = 0

        # Any exception inside this context rolls back the transaction.
        with psycopg.connect(database_url) as conn:
            with conn.cursor() as cur:
                cur.execute(
                    """
                    SELECT status, batch_type
                    FROM public.pipeline_batch_runs
                    WHERE batch_id = %s
                    FOR UPDATE
                    """,
                    (batch_id,),
                )
                batch = cur.fetchone()

                if batch is None:
                    raise RuntimeError("Batch is not registered")

                if batch[0] != "running":
                    raise RuntimeError(
                        f"Batch must be running, found status: {batch[0]}"
                    )

                if batch[1] != "incremental":
                    raise RuntimeError(
                        "Incremental writer requires an incremental batch"
                    )

                for record in records:
                    cur.execute(
                        """
                        INSERT INTO public.phase5_incremental_orders (
                            order_id,
                            customer_id,
                            order_status,
                            order_purchase_timestamp,
                            source_batch_id,
                            updated_at
                        )
                        VALUES (%s, %s, %s, %s, %s, NOW())
                        ON CONFLICT (order_id) DO UPDATE SET
                            customer_id = EXCLUDED.customer_id,
                            order_status = EXCLUDED.order_status,
                            order_purchase_timestamp =
                                EXCLUDED.order_purchase_timestamp,
                            source_batch_id = EXCLUDED.source_batch_id,
                            updated_at = NOW()
                        RETURNING (xmax = 0) AS was_inserted
                        """,
                        (
                            record["order_id"],
                            record["customer_id"],
                            record["order_status"],
                            record["order_purchase_timestamp"],
                            batch_id,
                        ),
                    )

                    if cur.fetchone()[0]:
                        inserted_rows += 1
                    else:
                        updated_rows += 1

                cur.execute(
                    """
                    UPDATE public.pipeline_batch_runs
                    SET status = 'succeeded',
                        finished_at = NOW(),
                        source_rows = %s,
                        inserted_rows = %s,
                        updated_rows = %s,
                        rejected_rows = 0,
                        error_message = NULL
                    WHERE batch_id = %s
                      AND status = 'running'
                    RETURNING batch_id
                    """,
                    (
                        len(records),
                        inserted_rows,
                        updated_rows,
                        batch_id,
                    ),
                )

                if cur.fetchone() is None:
                    raise RuntimeError(
                        "Could not mark batch successful"
                    )

        # Reaching here means the database transaction committed.
        return {
            "source_rows": len(records),
            "inserted_rows": inserted_rows,
            "updated_rows": updated_rows,
            "rejected_rows": 0,
        }

    except Exception as exc:
        # The write transaction has exited and rolled back before
        # this handler runs. Record failure using a separate transaction.
        try:
            mark_batch_failed(
                database_url,
                batch_id,
                f"{type(exc).__name__}: {exc}",
            )
        except Exception:
            # Preserve the original processing error if failure recording
            # also fails, for example because the database is unavailable.
            pass

        raise
'''

lines = source.splitlines(keepends=True)
start = writer_node.lineno - 1
end = writer_node.end_lineno

updated_source = (
    "".join(lines[:start])
    + replacement
    + "\n"
    + "".join(lines[end:])
)

# Validate before saving.
ast.parse(updated_source)
compile(updated_source, str(module_path), "exec")

module_path.write_text(updated_source, encoding="utf-8")

print("Updated write_incremental_orders() successfully.")
print("Python syntax validation: PASS")
print("Other functions were preserved.")

Updated write_incremental_orders() successfully.
Python syntax validation: PASS
Other functions were preserved.


Test transaction rollback and failure recording

In [43]:
import importlib
import psycopg
from datetime import datetime
from google.colab import userdata
import sales_pipeline.incremental as incremental

database_url = userdata.get("NEON_DB_URL")

# Reload the updated source code into the current Colab runtime.
incremental = importlib.reload(incremental)

print(
    "Updated writer loaded:",
    hasattr(incremental, "write_incremental_orders"),
)

# Repair the previous test batch, which was left running by the stale function.
previous_batch_id = incremental.make_batch_id(
    "phase5_failure_test",
    "duplicate-order-rollback-001",
)

incremental.mark_batch_failed(
    database_url,
    previous_batch_id,
    "Test recovery: previous run used a stale imported function.",
)

# Create a fresh test batch.
batch_id = incremental.make_batch_id(
    "phase5_failure_test",
    "duplicate-order-rollback-002",
)

status = incremental.register_batch(
    database_url,
    batch_id,
    batch_type="incremental",
)
print("New batch registration:", status)

records = [
    {
        "order_id": "phase5_rollback_order_002",
        "customer_id": "phase5_customer_001",
        "order_status": "processing",
        "order_purchase_timestamp": datetime(2026, 10, 17, 12, 0),
    },
    {
        "order_id": "phase5_rollback_order_002",
        "customer_id": "phase5_customer_002",
        "order_status": "shipped",
        "order_purchase_timestamp": datetime(2026, 10, 17, 13, 0),
    },
]

try:
    incremental.write_incremental_orders(
        database_url,
        batch_id,
        records,
    )
except ValueError as exc:
    print("Expected validation error:", exc)
else:
    raise AssertionError("Duplicate input was not rejected")

with psycopg.connect(database_url) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT status, error_message, inserted_rows
            FROM public.pipeline_batch_runs
            WHERE batch_id = %s
            """,
            (batch_id,),
        )
        batch_result = cur.fetchone()

        cur.execute(
            """
            SELECT COUNT(*)
            FROM public.phase5_incremental_orders
            WHERE order_id = %s
            """,
            ("phase5_rollback_order_002",),
        )
        target_count = cur.fetchone()[0]

print("Batch metadata:", batch_result)
print("Target rows:", target_count)

assert batch_result[0] == "failed"
assert batch_result[1] is not None
assert target_count == 0

print("PASS: the reloaded writer records the failed batch.")

Updated writer loaded: True
New batch registration: started
Expected validation error: Duplicate order_id within input batch: phase5_rollback_order_002
Batch metadata: ('failed', 'ValueError: Duplicate order_id within input batch: phase5_rollback_order_002', 0)
Target rows: 0
PASS: the reloaded writer records the failed batch.


Test a real database-write failure and rollback

In [44]:
import importlib
import psycopg
from datetime import datetime
from google.colab import userdata
import sales_pipeline.incremental as incremental

incremental = importlib.reload(incremental)
database_url = userdata.get("NEON_DB_URL")

batch_id = incremental.make_batch_id(
    "phase5_failure_test",
    "database-rollback-001",
)

registration = incremental.register_batch(
    database_url,
    batch_id,
    batch_type="incremental",
)
print("Batch registration:", registration)

if registration != "started":
    raise RuntimeError(
        f"Expected a new batch, got: {registration}"
    )

constraint_created = False

try:
    with psycopg.connect(database_url) as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                ALTER TABLE public.phase5_incremental_orders
                ADD CONSTRAINT phase5_test_reject_order
                CHECK (order_id <> 'phase5_force_db_failure')
                """
            )
    constraint_created = True

    records = [
        {
            "order_id": "phase5_rollback_should_not_exist",
            "customer_id": "phase5_customer_rollback",
            "order_status": "processing",
            "order_purchase_timestamp": datetime(
                2026, 10, 17, 14, 0
            ),
        },
        {
            "order_id": "phase5_force_db_failure",
            "customer_id": "phase5_customer_failure",
            "order_status": "processing",
            "order_purchase_timestamp": datetime(
                2026, 10, 17, 14, 1
            ),
        },
    ]

    try:
        incremental.write_incremental_orders(
            database_url,
            batch_id,
            records,
        )
    except Exception as exc:
        print("Expected database-write error:", type(exc).__name__)
    else:
        raise AssertionError(
            "The deliberate database constraint failure did not occur"
        )

finally:
    if constraint_created:
        with psycopg.connect(database_url) as conn:
            with conn.cursor() as cur:
                cur.execute(
                    """
                    ALTER TABLE public.phase5_incremental_orders
                    DROP CONSTRAINT IF EXISTS phase5_test_reject_order
                    """
                )

with psycopg.connect(database_url) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT status, error_message, inserted_rows
            FROM public.pipeline_batch_runs
            WHERE batch_id = %s
            """,
            (batch_id,),
        )
        metadata = cur.fetchone()

        cur.execute(
            """
            SELECT COUNT(*)
            FROM public.phase5_incremental_orders
            WHERE order_id = %s
            """,
            ("phase5_rollback_should_not_exist",),
        )
        partial_insert_count = cur.fetchone()[0]

print("Batch metadata:", metadata)
print("Partial insert count:", partial_insert_count)

assert metadata[0] == "failed"
assert metadata[1] is not None
assert partial_insert_count == 0

print("PASS: database failure rolled back the partial insert.")

Batch registration: started
Expected database-write error: CheckViolation
Batch metadata: ('failed', 'CheckViolation: new row for relation "phase5_incremental_orders" violates check constraint "phase5_test_reject_order"\nDETAIL:  Failing row contains (phase5_force_db_failure, phase5_customer_failure, processing, 2026-10-17 14:01:00, batch_f70a5d955c0c4e79ea343d976425fe2c, 2026-10-17 18:13:28.711448+00).', 0)
Partial insert count: 0
PASS: database failure rolled back the partial insert.


# Automated Tests and Execution Logging

Inspect the exact files we need to change

In [45]:
from pathlib import Path

files_to_check = [
    PROJECT_ROOT / "src" / "sales_pipeline" / "incremental.py",
    PROJECT_ROOT / "tests" / "test_incremental.py",
]

for file_path in files_to_check:
    print(f"\n{'=' * 20} {file_path.name} {'=' * 20}")

    if not file_path.exists():
        print("FILE NOT FOUND")
        continue

    content = file_path.read_text(encoding="utf-8")
    print(content)


==================== incremental.py ====================
"""Batch tracking and lifecycle management for incremental processing."""

import hashlib
import json

import psycopg


BATCH_TABLE = "public.pipeline_batch_runs"


def make_batch_id(source_name: str, batch_key: str) -> str:
    """Return a deterministic ID for a logical source batch."""
    if not isinstance(source_name, str) or not source_name.strip():
        raise ValueError("source_name cannot be empty")
    if not isinstance(batch_key, str) or not batch_key.strip():
        raise ValueError("batch_key cannot be empty")

    identity = json.dumps(
        {
            "source": source_name.strip(),
            "batch_key": batch_key.strip(),
        },
        sort_keys=True,
        separators=(",", ":"),
    )
    digest = hashlib.sha256(identity.encode("utf-8")).hexdigest()
    return f"batch_{digest[:32]}"


def register_batch(
    database_url: str,
    batch_id: str,
    batch_type: str = "incremental",
) -> str:
   

In [47]:
import ast
from pathlib import Path

module_path = (
    PROJECT_ROOT / "src" / "sales_pipeline" / "incremental.py"
)
source = module_path.read_text(encoding="utf-8")
tree = ast.parse(source)

# Add the logging import if it is not already present.
if "import logging" not in source:
    source = source.replace(
        "import json",
        "import json\nimport logging",
        1,
    )

# Add the module logger if it is not already present.
if "logger = logging.getLogger(__name__)" not in source:
    source = source.replace(
        'BATCH_TABLE = "public.pipeline_batch_runs"',
        'BATCH_TABLE = "public.pipeline_batch_runs"\n'
        '\nlogger = logging.getLogger(__name__)',
        1,
    )

tree = ast.parse(source)
lines = source.splitlines(keepends=True)

# Insert log statements at the end of selected functions,
# preserving all existing statements.
insertions = []

for node in tree.body:
    if not isinstance(node, ast.FunctionDef):
        continue

    if node.name == "register_batch":
        # Logging is added at individual lifecycle branches in the next step.
        continue

    if node.name == "mark_batch_succeeded":
        insertions.append((
            node.end_lineno,
            '\n    logger.info(\n'
            '        "Batch succeeded: batch_id=%s source_rows=%s "\n'
            '        "inserted_rows=%s updated_rows=%s rejected_rows=%s",\n'
            '        batch_id, source_rows, inserted_rows, updated_rows, rejected_rows,\n'
            '    )\n'
        ))

    elif node.name == "mark_batch_failed":
        insertions.append((
            node.end_lineno,
            '\n    logger.error(\n'
            '        "Batch failed: batch_id=%s error=%s",\n'
            '        batch_id, error_message[:500],\n'
            '    )\n'
        ))

    elif node.name == "write_incremental_orders":
        # Log success immediately before the return statement.
        for child in ast.walk(node):
            if isinstance(child, ast.Return) and child.lineno > node.lineno:
                if (
                    isinstance(child.value, ast.Dict)
                    and child.lineno > 300
                ):
                    insertions.append((
                        child.lineno - 1,
                        '        logger.info(\n'
                        '            "Incremental write committed: batch_id=%s "\n'
                        '            "source_rows=%s inserted_rows=%s updated_rows=%s",\n'
                        '            batch_id, len(records), inserted_rows, updated_rows,\n'
                        '        )\n'
                    ))
                    break

        # Log an exception before attempting failure recording.
        for child in ast.walk(node):
            if isinstance(child, ast.ExceptHandler):
                if child.name == "exc":
                    insertions.append((
                        child.body[0].lineno - 1,
                        '        logger.exception(\n'
                        '            "Incremental processing failed: batch_id=%s",\n'
                        '            batch_id,\n'
                        '        )\n'
                    ))
                    break

# Apply insertions from bottom to top to keep line positions stable.
for line_number, text_to_insert in sorted(
    insertions, key=lambda item: item[0], reverse=True
):
    lines.insert(line_number, text_to_insert)

updated = "".join(lines)

# Do not save if parsing or compilation fails.
ast.parse(updated)
compile(updated, str(module_path), "exec")

module_path.write_text(updated, encoding="utf-8")

print("Logging changes saved.")
print("Syntax validation: PASS")
print("Existing functions preserved.")

Logging changes saved.
Syntax validation: PASS
Existing functions preserved.


Run a logging smoke test

In [48]:
import importlib
import logging
import uuid
from google.colab import userdata
import sales_pipeline.incremental as incremental

# Configure visible INFO-level logs for this notebook.
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(name)s | %(message)s",
    force=True,
)

incremental = importlib.reload(incremental)
database_url = userdata.get("NEON_DB_URL")

batch_id = incremental.make_batch_id(
    "phase5_logging_test",
    f"logging-smoke-{uuid.uuid4()}",
)

status = incremental.register_batch(
    database_url,
    batch_id,
    batch_type="incremental",
)
print("Registration result:", status)

if status == "started":
    incremental.mark_batch_succeeded(
        database_url,
        batch_id,
        source_rows=3,
        inserted_rows=2,
        updated_rows=1,
        rejected_rows=0,
    )

rerun_status = incremental.register_batch(
    database_url,
    batch_id,
    batch_type="incremental",
)
print("Completed rerun result:", rerun_status)

assert status == "started"
assert rerun_status == "skipped"

print("PASS: logging smoke test completed.")

Registration result: started


INFO | sales_pipeline.incremental | Batch succeeded: batch_id=batch_31e1781bed554866ff6999898e8cc73c source_rows=3 inserted_rows=2 updated_rows=1 rejected_rows=0


Completed rerun result: skipped
PASS: logging smoke test completed.


Add the missing registration and retry logs

In [49]:
import ast
from pathlib import Path

module_path = (
    PROJECT_ROOT / "src" / "sales_pipeline" / "incremental.py"
)
source = module_path.read_text(encoding="utf-8")

old_new_batch = '''            if cur.fetchone() is not None:
                return "started"'''

new_new_batch = '''            if cur.fetchone() is not None:
                logger.info(
                    "Batch registered: batch_id=%s type=%s attempt=1",
                    batch_id,
                    batch_type,
                )
                return "started"'''

old_skip = '''            if status == "succeeded":
                return "skipped"'''

new_skip = '''            if status == "succeeded":
                logger.info(
                    "Batch skipped because it already succeeded: batch_id=%s",
                    batch_id,
                )
                return "skipped"'''

old_retry = '''                return "started"

            raise RuntimeError(f"Unexpected batch status: {status}")'''

new_retry = '''                logger.info(
                    "Batch retry started: batch_id=%s attempt=%s",
                    batch_id,
                    attempt_number + 1,
                )
                return "started"

            raise RuntimeError(f"Unexpected batch status: {status}")'''

for old, new, label in [
    (old_new_batch, new_new_batch, "new batch"),
    (old_skip, new_skip, "completed batch skip"),
    (old_retry, new_retry, "retry"),
]:
    count = source.count(old)
    if count != 1:
        raise RuntimeError(
            f"Expected one matching {label} branch; found {count}. "
            "No changes were saved."
        )
    source = source.replace(old, new, 1)

ast.parse(source)
compile(source, str(module_path), "exec")
module_path.write_text(source, encoding="utf-8")

print("Registration, skip, and retry logging updated.")
print("Syntax validation: PASS")

Registration, skip, and retry logging updated.
Syntax validation: PASS


Verify all lifecycle logs

In [50]:
import importlib
import logging
import uuid
from google.colab import userdata
import sales_pipeline.incremental as incremental

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(name)s | %(message)s",
    force=True,
)

incremental = importlib.reload(incremental)
database_url = userdata.get("NEON_DB_URL")

batch_id = incremental.make_batch_id(
    "phase5_logging_test",
    f"lifecycle-{uuid.uuid4()}",
)

# Register a new batch.
first_status = incremental.register_batch(
    database_url,
    batch_id,
    batch_type="incremental",
)
print("First registration:", first_status)

# Complete it and verify success logging.
incremental.mark_batch_succeeded(
    database_url,
    batch_id,
    source_rows=3,
    inserted_rows=2,
    updated_rows=1,
    rejected_rows=0,
)

# Re-register the successful batch to verify skip logging.
second_status = incremental.register_batch(
    database_url,
    batch_id,
    batch_type="incremental",
)
print("Second registration:", second_status)

assert first_status == "started"
assert second_status == "skipped"

print("PASS: lifecycle logging smoke test.")

INFO | sales_pipeline.incremental | Batch registered: batch_id=batch_bdda4628c720612ca5e791b61cb1f58d type=incremental attempt=1


First registration: started


INFO | sales_pipeline.incremental | Batch succeeded: batch_id=batch_bdda4628c720612ca5e791b61cb1f58d source_rows=3 inserted_rows=2 updated_rows=1 rejected_rows=0
INFO | sales_pipeline.incremental | Batch skipped because it already succeeded: batch_id=batch_bdda4628c720612ca5e791b61cb1f58d


Second registration: skipped
PASS: lifecycle logging smoke test.


Add automated tests for writer validation

In [51]:
from pathlib import Path
import ast
import subprocess
import sys

test_path = PROJECT_ROOT / "tests" / "test_incremental.py"
source = test_path.read_text(encoding="utf-8")

if "TestIncrementalWriterValidation" not in source:
    additions = '''

class TestIncrementalWriterValidation(unittest.TestCase):
    """Validate records before any database write is attempted."""

    def setUp(self):
        from unittest.mock import patch
        self.connect_patcher = patch(
            "sales_pipeline.incremental.psycopg.connect",
            side_effect=AssertionError(
                "Database connection should not be attempted"
            ),
        )
        self.connect_patcher.start()
        self.addCleanup(self.connect_patcher.stop)

    def test_non_list_records_rejected(self):
        from sales_pipeline.incremental import write_incremental_orders

        with self.assertRaises(TypeError):
            write_incremental_orders("unused", "batch_test", None)

    def test_non_dictionary_record_rejected(self):
        from sales_pipeline.incremental import write_incremental_orders

        with self.assertRaisesRegex(TypeError, "must be a dictionary"):
            write_incremental_orders("unused", "batch_test", ["invalid"])

    def test_missing_fields_rejected(self):
        from sales_pipeline.incremental import write_incremental_orders

        with self.assertRaisesRegex(ValueError, "missing fields"):
            write_incremental_orders(
                "unused",
                "batch_test",
                [{"order_id": "order_1"}],
            )

    def test_blank_order_id_rejected(self):
        from sales_pipeline.incremental import write_incremental_orders

        record = {
            "order_id": " ",
            "customer_id": "customer_1",
            "order_status": "processing",
            "order_purchase_timestamp": None,
        }
        with self.assertRaisesRegex(ValueError, "invalid order_id"):
            write_incremental_orders("unused", "batch_test", [record])

    def test_blank_customer_id_rejected(self):
        from sales_pipeline.incremental import write_incremental_orders

        record = {
            "order_id": "order_1",
            "customer_id": " ",
            "order_status": "processing",
            "order_purchase_timestamp": None,
        }
        with self.assertRaisesRegex(ValueError, "invalid customer_id"):
            write_incremental_orders("unused", "batch_test", [record])

    def test_duplicate_order_ids_rejected(self):
        from sales_pipeline.incremental import write_incremental_orders

        record = {
            "order_id": "order_1",
            "customer_id": "customer_1",
            "order_status": "processing",
            "order_purchase_timestamp": None,
        }
        with self.assertRaisesRegex(ValueError, "Duplicate order_id"):
            write_incremental_orders(
                "unused",
                "batch_test",
                [record, record.copy()],
            )
'''

    updated_source = source.rstrip() + "\n" + additions + "\n"
    ast.parse(updated_source)
    test_path.write_text(updated_source, encoding="utf-8")
    print("Added six writer-validation tests.")
else:
    print("Writer-validation tests already exist; no changes made.")

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "tests.test_incremental",
        "-v",
    ],
    cwd=str(PROJECT_ROOT),
    text=True,
    capture_output=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Unit tests failed; inspect the output above.")

print("PASS: incremental unit tests.")

Added six writer-validation tests.

test_batch_id_has_expected_prefix_and_length (tests.test_incremental.TestBatchIdentity.test_batch_id_has_expected_prefix_and_length) ... ok
test_different_batch_keys_produce_different_ids (tests.test_incremental.TestBatchIdentity.test_different_batch_keys_produce_different_ids) ... ok
test_different_sources_produce_different_ids (tests.test_incremental.TestBatchIdentity.test_different_sources_produce_different_ids) ... ok
test_empty_batch_key_is_rejected (tests.test_incremental.TestBatchIdentity.test_empty_batch_key_is_rejected) ... ok
test_empty_source_is_rejected (tests.test_incremental.TestBatchIdentity.test_empty_source_is_rejected) ... ok
test_same_identity_produces_same_id (tests.test_incremental.TestBatchIdentity.test_same_identity_produces_same_id) ... ok
test_whitespace_is_normalized (tests.test_incremental.TestBatchIdentity.test_whitespace_is_normalized) ... ok
test_blank_customer_id_rejected (tests.test_incremental.TestIncrementalWriterVal

Run integration and regression tests

In [52]:
import subprocess
import sys
from pathlib import Path

test_files = [
    "tests.test_incremental",
    "tests.test_incremental_integration",
    "tests.test_pipeline",
    "tests.test_phase4_storage_modeling",
    "tests.test_spark_transformations",
]

results = {}

for test_module in test_files:
    print(f"\n{'=' * 12} {test_module} {'=' * 12}")

    result = subprocess.run(
        [sys.executable, "-m", "unittest", test_module, "-v"],
        cwd=str(PROJECT_ROOT),
        text=True,
        capture_output=True,
    )

    print(result.stdout)
    if result.stderr:
        print(result.stderr)

    results[test_module] = result.returncode == 0

print("\nREGRESSION SUMMARY")
for test_module, passed in results.items():
    print(f"{'PASS' if passed else 'FAIL'}: {test_module}")

if not all(results.values()):
    raise RuntimeError(
        "Some test modules failed. Review the output before proceeding."
    )

print("\nPASS: all selected regression test modules.")


============ tests.test_incremental ============

test_batch_id_has_expected_prefix_and_length (tests.test_incremental.TestBatchIdentity.test_batch_id_has_expected_prefix_and_length) ... ok
test_different_batch_keys_produce_different_ids (tests.test_incremental.TestBatchIdentity.test_different_batch_keys_produce_different_ids) ... ok
test_different_sources_produce_different_ids (tests.test_incremental.TestBatchIdentity.test_different_sources_produce_different_ids) ... ok
test_empty_batch_key_is_rejected (tests.test_incremental.TestBatchIdentity.test_empty_batch_key_is_rejected) ... ok
test_empty_source_is_rejected (tests.test_incremental.TestBatchIdentity.test_empty_source_is_rejected) ... ok
test_same_identity_produces_same_id (tests.test_incremental.TestBatchIdentity.test_same_identity_produces_same_id) ... ok
test_whitespace_is_normalized (tests.test_incremental.TestBatchIdentity.test_whitespace_is_normalized) ... ok
test_blank_customer_id_rejected (tests.test_incremental.TestIncre

RuntimeError: Some test modules failed. Review the output before proceeding.

Rerun the Spark test with the source path configured

In [53]:
import os
import subprocess
import sys

project_root = str(PROJECT_ROOT)
src_path = str(PROJECT_ROOT / "src")

env = os.environ.copy()
existing_pythonpath = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = (
    src_path + os.pathsep + existing_pythonpath
    if existing_pythonpath
    else src_path
)

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "tests.test_spark_transformations",
        "-v",
    ],
    cwd=project_root,
    env=env,
    text=True,
    capture_output=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Spark transformation tests failed.")

print("PASS: Spark transformation regression tests.")

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/17 18:23:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes wher

Run Neon integration tests with the secret passed safely

In [54]:
import os
import subprocess
import sys
from google.colab import userdata

database_url = userdata.get("NEON_DB_URL")
if not database_url:
    raise RuntimeError("NEON_DB_URL is missing from Colab Secrets.")

env = os.environ.copy()
env["PYTHONPATH"] = (
    str(PROJECT_ROOT / "src")
    + os.pathsep
    + env.get("PYTHONPATH", "")
)
env["NEON_DB_URL"] = database_url

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "tests.test_incremental_integration",
        "-v",
    ],
    cwd=str(PROJECT_ROOT),
    env=env,
    text=True,
    capture_output=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Neon incremental integration tests failed.")

if "skipped=" in result.stdout:
    raise RuntimeError(
        "Integration tests were skipped; inspect the output."
    )

print("PASS: Neon integration tests executed successfully.")


test_failed_batch_can_be_retried (tests.test_incremental_integration.TestIncrementalBatchIntegration.test_failed_batch_can_be_retried) ... Batch failed: batch_id=batch_0c0dd83c396dee9999ed89801376870d error=Controlled integration-test failure
ok
test_successful_batch_rerun_is_skipped (tests.test_incremental_integration.TestIncrementalBatchIntegration.test_successful_batch_rerun_is_skipped) ... ok

----------------------------------------------------------------------
Ran 2 tests in 15.325s

OK

PASS: Neon integration tests executed successfully.


Final consolidated regression suite

In [55]:
import os
import subprocess
import sys
from google.colab import userdata

database_url = userdata.get("NEON_DB_URL")
if not database_url:
    raise RuntimeError("NEON_DB_URL is missing from Colab Secrets.")

env = os.environ.copy()
env["NEON_DB_URL"] = database_url
env["PYTHONPATH"] = (
    str(PROJECT_ROOT / "src")
    + os.pathsep
    + env.get("PYTHONPATH", "")
)

test_modules = [
    "tests.test_incremental",
    "tests.test_incremental_integration",
    "tests.test_pipeline",
    "tests.test_phase4_storage_modeling",
    "tests.test_spark_transformations",
]

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        *test_modules,
        "-v",
    ],
    cwd=str(PROJECT_ROOT),
    env=env,
    text=True,
    capture_output=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "Consolidated regression suite failed. "
        "Review the failing test output before proceeding."
    )

if "skipped=" in result.stdout:
    raise RuntimeError(
        "At least one test was skipped. Check the test output."
    )

print("\nPASS: consolidated regression suite completed.")
print("Phase 4 regression coverage remains green.")
print("No Git commit or push was performed.")

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SALES ANALYTICS DATA PIPELINE

[1/5] Validating source data...

[2/5] Building order analytics...

[3/5] Building product and category analytics...

[4/5] Building customer analytics...

[5/5] Building sales performance analytics...

PIPELINE COMPLETED SUCCESSFULLY
Duration: 0.0 seconds
Validation checks: 1
Order rows: 2
Product rows: 1
Cat

Create the rejected-records quarantine table

In [56]:
import psycopg
from google.colab import userdata

database_url = userdata.get("NEON_DB_URL")
if not database_url:
    raise RuntimeError("NEON_DB_URL is missing from Colab Secrets.")

with psycopg.connect(database_url) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS public.phase5_rejected_records (
                quarantine_id BIGSERIAL PRIMARY KEY,
                batch_id TEXT NOT NULL
                    REFERENCES public.pipeline_batch_runs(batch_id),
                record_index INTEGER NOT NULL
                    CHECK (record_index >= 0),
                record_payload JSONB NOT NULL,
                rejection_reason TEXT NOT NULL,
                created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
                CONSTRAINT uq_phase5_rejected_batch_record
                    UNIQUE (batch_id, record_index)
            )
        """)

        cur.execute("""
            SELECT column_name, data_type
            FROM information_schema.columns
            WHERE table_schema = 'public'
              AND table_name = 'phase5_rejected_records'
            ORDER BY ordinal_position
        """)
        columns = cur.fetchall()

print("Quarantine table verified:")
for column_name, data_type in columns:
    print(f"- {column_name}: {data_type}")

assert len(columns) == 6
print("PASS: quarantine schema is ready.")

Quarantine table verified:
- quarantine_id: bigint
- batch_id: text
- record_index: integer
- record_payload: jsonb
- rejection_reason: text
- created_at: timestamp with time zone
PASS: quarantine schema is ready.


Implement record validation and quarantine

In [57]:
import ast
from pathlib import Path

module_path = (
    PROJECT_ROOT / "src" / "sales_pipeline" / "incremental.py"
)
source = module_path.read_text(encoding="utf-8")

if "def validate_and_quarantine_orders(" in source:
    print("validate_and_quarantine_orders() already exists; no changes made.")
else:
    helper = '''

def validate_and_quarantine_orders(
    database_url: str,
    batch_id: str,
    records: list[dict],
) -> dict:
    """Split order records into valid and rejected records.

    Rejected records are stored in the Phase 5 quarantine table.
    The batch must already be registered.
    """
    import json

    if not isinstance(records, list):
        raise TypeError("records must be a list of dictionaries")

    required_fields = {
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
    }

    valid_records = []
    rejected_records = []
    seen_ids = set()

    for index, record in enumerate(records):
        reason = None

        if not isinstance(record, dict):
            reason = "Record must be a dictionary"
        else:
            missing = required_fields - record.keys()
            if missing:
                reason = f"Missing required fields: {sorted(missing)}"
            elif (
                not isinstance(record["order_id"], str)
                or not record["order_id"].strip()
            ):
                reason = "Invalid order_id"
            elif (
                not isinstance(record["customer_id"], str)
                or not record["customer_id"].strip()
            ):
                reason = "Invalid customer_id"
            elif record["order_id"] in seen_ids:
                reason = f"Duplicate order_id within batch: {record['order_id']}"

        if reason:
            payload = record if isinstance(record, dict) else {
                "_invalid_record_type": type(record).__name__,
                "_value": repr(record),
            }
            rejected_records.append({
                "record_index": index,
                "record_payload": payload,
                "rejection_reason": reason,
            })
        else:
            seen_ids.add(record["order_id"])
            valid_records.append(record)

    # Persist quarantine records before returning valid records.
    # The uniqueness constraint makes repeated quarantine attempts safe
    # for the same batch and source record index.
    if rejected_records:
        with psycopg.connect(database_url) as conn:
            with conn.cursor() as cur:
                cur.execute(
                    """
                    SELECT status
                    FROM public.pipeline_batch_runs
                    WHERE batch_id = %s
                    FOR UPDATE
                    """,
                    (batch_id,),
                )
                batch = cur.fetchone()

                if batch is None or batch[0] != "running":
                    raise RuntimeError(
                        "Quarantine requires a registered running batch"
                    )

                for rejected in rejected_records:
                    cur.execute(
                        """
                        INSERT INTO public.phase5_rejected_records (
                            batch_id,
                            record_index,
                            record_payload,
                            rejection_reason
                        )
                        VALUES (%s, %s, %s::jsonb, %s)
                        ON CONFLICT (batch_id, record_index) DO NOTHING
                        """,
                        (
                            batch_id,
                            rejected["record_index"],
                            json.dumps(
                                rejected["record_payload"],
                                default=str,
                            ),
                            rejected["rejection_reason"],
                        ),
                    )

        logger.warning(
            "Order records quarantined: batch_id=%s rejected_rows=%s",
            batch_id,
            len(rejected_records),
        )

    logger.info(
        "Order validation completed: batch_id=%s source_rows=%s "
        "valid_rows=%s rejected_rows=%s",
        batch_id,
        len(records),
        len(valid_records),
        len(rejected_records),
    )

    return {
        "valid_records": valid_records,
        "rejected_records": rejected_records,
        "source_rows": len(records),
        "valid_rows": len(valid_records),
        "rejected_rows": len(rejected_records),
    }
'''

    updated = source.rstrip() + "\n" + helper + "\n"
    ast.parse(updated)
    compile(updated, str(module_path), "exec")
    module_path.write_text(updated, encoding="utf-8")
    print("Added validate_and_quarantine_orders().")
    print("Python syntax validation: PASS")

Added validate_and_quarantine_orders().
Python syntax validation: PASS


Test quarantine behavior in Neon

In [58]:
import importlib
import logging
import uuid
import psycopg
from google.colab import userdata
import sales_pipeline.incremental as incremental

logging.basicConfig(level=logging.INFO, force=True)
incremental = importlib.reload(incremental)

database_url = userdata.get("NEON_DB_URL")
batch_id = incremental.make_batch_id(
    "phase5_quarantine_test",
    f"quarantine-{uuid.uuid4()}",
)

status = incremental.register_batch(
    database_url,
    batch_id,
    batch_type="incremental",
)
print("Batch registration:", status)

if status != "started":
    raise RuntimeError(f"Expected started, got {status}")

records = [
    {
        "order_id": "phase5_quarantine_valid_001",
        "customer_id": "phase5_customer_001",
        "order_status": "processing",
        "order_purchase_timestamp": "2026-10-17 12:00:00",
    },
    {
        "order_id": "",
        "customer_id": "phase5_customer_002",
        "order_status": "processing",
        "order_purchase_timestamp": "2026-10-17 12:01:00",
    },
    {
        "order_id": "phase5_quarantine_valid_001",
        "customer_id": "phase5_customer_003",
        "order_status": "shipped",
        "order_purchase_timestamp": "2026-10-17 12:02:00",
    },
    {
        "order_id": "phase5_quarantine_invalid_004",
        "order_status": "processing",
        "order_purchase_timestamp": "2026-10-17 12:03:00",
    },
]

result = incremental.validate_and_quarantine_orders(
    database_url,
    batch_id,
    records,
)

print(
    "Validation counts:",
    {
        "source": result["source_rows"],
        "valid": result["valid_rows"],
        "rejected": result["rejected_rows"],
    },
)
print("Rejection reasons:", [
    item["rejection_reason"] for item in result["rejected_records"]
])

# Repeat validation to verify quarantine deduplication.
incremental.validate_and_quarantine_orders(
    database_url,
    batch_id,
    records,
)

with psycopg.connect(database_url) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT COUNT(*)
            FROM public.phase5_rejected_records
            WHERE batch_id = %s
            """,
            (batch_id,),
        )
        quarantine_count = cur.fetchone()[0]

print("Quarantine rows after two calls:", quarantine_count)

assert result["source_rows"] == 4
assert result["valid_rows"] == 1
assert result["rejected_rows"] == 3
assert quarantine_count == 3

print("PASS: valid records separated and rejected records quarantined once.")

INFO:sales_pipeline.incremental:Batch registered: batch_id=batch_855a90d879329792cd9eaa767baf6c29 type=incremental attempt=1


Batch registration: started


INFO:sales_pipeline.incremental:Order validation completed: batch_id=batch_855a90d879329792cd9eaa767baf6c29 source_rows=4 valid_rows=1 rejected_rows=3


Validation counts: {'source': 4, 'valid': 1, 'rejected': 3}
Rejection reasons: ['Invalid order_id', 'Duplicate order_id within batch: phase5_quarantine_valid_001', "Missing required fields: ['customer_id']"]


INFO:sales_pipeline.incremental:Order validation completed: batch_id=batch_855a90d879329792cd9eaa767baf6c29 source_rows=4 valid_rows=1 rejected_rows=3


Quarantine rows after two calls: 3
PASS: valid records separated and rejected records quarantined once.


Add bounded retries for transient database errors

In [59]:
import ast
from pathlib import Path

module_path = (
    PROJECT_ROOT / "src" / "sales_pipeline" / "incremental.py"
)
source = module_path.read_text(encoding="utf-8")

if "def retry_transient_database_operation(" in source:
    print("Retry helper already exists; no changes made.")
else:
    helper = '''

def retry_transient_database_operation(
    operation,
    *,
    max_attempts: int = 3,
    initial_delay_seconds: float = 1.0,
    max_delay_seconds: float = 8.0,
):
    """Retry an operation on transient PostgreSQL errors only.

    Uses exponential backoff. The operation must be safe to retry.
    """
    import random
    import time
    import psycopg

    if not callable(operation):
        raise TypeError("operation must be callable")
    if not isinstance(max_attempts, int) or isinstance(max_attempts, bool):
        raise ValueError("max_attempts must be an integer")
    if max_attempts < 1:
        raise ValueError("max_attempts must be at least 1")
    if initial_delay_seconds < 0 or max_delay_seconds < 0:
        raise ValueError("Retry delays cannot be negative")

    transient_errors = (
        psycopg.OperationalError,
        psycopg.errors.SerializationFailure,
        psycopg.errors.DeadlockDetected,
        psycopg.errors.LockNotAvailable,
    )

    for attempt in range(1, max_attempts + 1):
        try:
            return operation()
        except transient_errors as exc:
            if attempt == max_attempts:
                logger.error(
                    "Database operation exhausted retries: attempts=%s error_type=%s",
                    attempt,
                    type(exc).__name__,
                )
                raise

            delay = min(
                initial_delay_seconds * (2 ** (attempt - 1)),
                max_delay_seconds,
            )
            # Small jitter helps avoid synchronized retries.
            delay = delay * random.uniform(0.8, 1.2)

            logger.warning(
                "Transient database error: attempt=%s/%s error_type=%s "
                "retry_delay_seconds=%.2f",
                attempt,
                max_attempts,
                type(exc).__name__,
                delay,
            )
            time.sleep(delay)
'''

    updated = source.rstrip() + "\n" + helper + "\n"
    ast.parse(updated)
    compile(updated, str(module_path), "exec")
    module_path.write_text(updated, encoding="utf-8")
    print("Added retry_transient_database_operation().")
    print("Python syntax validation: PASS")

Added retry_transient_database_operation().
Python syntax validation: PASS


Test bounded retries without connecting to Neon

In [60]:
import importlib
import logging
from unittest.mock import Mock, patch
import psycopg
import sales_pipeline.incremental as incremental

logging.basicConfig(level=logging.INFO, force=True)
incremental = importlib.reload(incremental)

# Test 1: transient failure, then success.
operation = Mock(
    side_effect=[
        psycopg.OperationalError("simulated temporary connection error"),
        "success",
    ]
)

with patch("time.sleep") as sleep_mock:
    result = incremental.retry_transient_database_operation(
        operation,
        max_attempts=3,
        initial_delay_seconds=0,
        max_delay_seconds=0,
    )

assert result == "success"
assert operation.call_count == 2
assert sleep_mock.call_count == 1
print("PASS: transient error retried once, then succeeded.")

# Test 2: permanent error must not be retried.
permanent_operation = Mock(
    side_effect=ValueError("invalid input")
)

try:
    incremental.retry_transient_database_operation(
        permanent_operation,
        max_attempts=3,
        initial_delay_seconds=0,
        max_delay_seconds=0,
    )
except ValueError as exc:
    assert str(exc) == "invalid input"
else:
    raise AssertionError("Expected permanent error was not raised")

assert permanent_operation.call_count == 1
print("PASS: permanent error was not retried.")

# Test 3: transient error exhausts the configured attempts.
exhausted_operation = Mock(
    side_effect=psycopg.OperationalError("still unavailable")
)

with patch("time.sleep"):
    try:
        incremental.retry_transient_database_operation(
            exhausted_operation,
            max_attempts=3,
            initial_delay_seconds=0,
            max_delay_seconds=0,
        )
    except psycopg.OperationalError:
        pass
    else:
        raise AssertionError("Expected exhausted retry error")

assert exhausted_operation.call_count == 3
print("PASS: retry limit enforced at three attempts.")

ERROR:sales_pipeline.incremental:Database operation exhausted retries: attempts=3 error_type=OperationalError


PASS: transient error retried once, then succeeded.
PASS: permanent error was not retried.
PASS: retry limit enforced at three attempts.


Add a retry wrapper for the incremental writer

In [61]:
import ast
from pathlib import Path

module_path = (
    PROJECT_ROOT / "src" / "sales_pipeline" / "incremental.py"
)
source = module_path.read_text(encoding="utf-8")

if "def write_incremental_orders_with_retry(" in source:
    print("Retry wrapper already exists; no changes made.")
else:
    wrapper = '''

def write_incremental_orders_with_retry(
    database_url: str,
    batch_id: str,
    records: list[dict],
    *,
    max_attempts: int = 3,
    initial_delay_seconds: float = 1.0,
    max_delay_seconds: float = 8.0,
) -> dict[str, int]:
    """Retry incremental writes after transient database errors.

    Each attempt invokes the transactional writer anew. The batch must
    be registered or retryable before the first attempt.
    """
    return retry_transient_database_operation(
        lambda: write_incremental_orders(
            database_url,
            batch_id,
            records,
        ),
        max_attempts=max_attempts,
        initial_delay_seconds=initial_delay_seconds,
        max_delay_seconds=max_delay_seconds,
    )
'''

    updated = source.rstrip() + "\n" + wrapper + "\n"
    ast.parse(updated)
    compile(updated, str(module_path), "exec")
    module_path.write_text(updated, encoding="utf-8")

    print("Added write_incremental_orders_with_retry().")
    print("Python syntax validation: PASS")

Added write_incremental_orders_with_retry().
Python syntax validation: PASS


In [64]:
import ast
import re
from pathlib import Path

# 1. Locate the existing project file
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

source_path = PROJECT_ROOT / "src" / "sales_pipeline" / "incremental.py"

if not source_path.exists():
    raise FileNotFoundError(f"Source file not found: {source_path}")

source = source_path.read_text(encoding="utf-8")

# 2. Add the internal error-handling option to the writer signature
header_pattern = re.compile(
    r"def write_incremental_orders\((.*?)\)\s*->\s*dict\[str, int\]:",
    re.DOTALL,
)

match = header_pattern.search(source)

if not match:
    raise RuntimeError(
        "Could not identify write_incremental_orders(). "
        "No file changes have been saved."
    )

if "_mark_failed_on_error" not in match.group(1):
    params = match.group(1).rstrip().rstrip(",")
    new_header = (
        f"def write_incremental_orders({params}, *, "
        "_mark_failed_on_error: bool = True) -> dict[str, int]:"
    )
    source = source[:match.start()] + new_header + source[match.end():]

# 3. Replace only the outer exception handler in the writer
tree = ast.parse(source)

writer = next(
    (
        node for node in tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == "write_incremental_orders"
    ),
    None,
)

if writer is None:
    raise RuntimeError("Writer function not found. No changes saved.")

handlers = [
    node
    for node in ast.walk(writer)
    if isinstance(node, ast.ExceptHandler)
    and isinstance(node.type, ast.Name)
    and node.type.id == "Exception"
]

if not handlers:
    raise RuntimeError(
        "No Exception handler found in writer. No changes saved."
    )

# The outer handler starts before its nested handler.
handler = min(handlers, key=lambda node: node.lineno)

lines = source.splitlines(keepends=True)
indent = re.match(r"\s*", lines[handler.lineno - 1]).group(0)

replacement = [
    f"{indent}except Exception as exc:\n",
    f"{indent}    logger.exception(\n",
    f'{indent}        "Incremental write failed for batch %s", batch_id\n',
    f"{indent}    )\n",
    f"{indent}    if _mark_failed_on_error:\n",
    f"{indent}        try:\n",
    f"{indent}            mark_batch_failed(\n",
    f"{indent}                database_url,\n",
    f"{indent}                batch_id,\n",
    f'{indent}                f"{{type(exc).__name__}}: {{exc}}",\n',
    f"{indent}            )\n",
    f"{indent}        except Exception:\n",
    f"{indent}            pass\n",
    f"{indent}    raise\n",
]

lines[handler.lineno - 1:handler.end_lineno] = replacement
source = "".join(lines)

# 4. Replace the retry wrapper
tree = ast.parse(source)

wrapper = next(
    (
        node for node in tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == "write_incremental_orders_with_retry"
    ),
    None,
)

if wrapper is None:
    raise RuntimeError(
        "Retry wrapper not found. No changes saved."
    )

lines = source.splitlines(keepends=True)
wrapper_indent = re.match(
    r"\s*", lines[wrapper.lineno - 1]
).group(0)

new_wrapper = '''def write_incremental_orders_with_retry(
    database_url: str,
    batch_id: str,
    records: list[dict],
    *,
    max_attempts: int = 3,
    initial_delay_seconds: float = 1.0,
    max_delay_seconds: float = 8.0,
) -> dict[str, int]:
    """Retry transient write errors and mark failure after retries are exhausted."""
    try:
        return retry_transient_database_operation(
            lambda: write_incremental_orders(
                database_url,
                batch_id,
                records,
                _mark_failed_on_error=False,
            ),
            max_attempts=max_attempts,
            initial_delay_seconds=initial_delay_seconds,
            max_delay_seconds=max_delay_seconds,
        )
    except Exception as exc:
        try:
            mark_batch_failed(
                database_url,
                batch_id,
                f"{type(exc).__name__}: {exc}",
            )
        except Exception:
            logger.exception(
                "Could not mark batch %s as failed after retries", batch_id
            )
        raise
'''

new_wrapper_lines = [
    wrapper_indent + line if line.strip() else line
    for line in new_wrapper.splitlines(keepends=True)
]

lines[wrapper.lineno - 1:wrapper.end_lineno] = new_wrapper_lines
updated_source = "".join(lines)

# 5. Validate the complete updated source before writing
ast.parse(updated_source)

# Save the updated file
source_path.write_text(updated_source, encoding="utf-8")

# 6. Verify the saved version
saved_source = source_path.read_text(encoding="utf-8")
ast.parse(saved_source)

saved_tree = ast.parse(saved_source)
function_names = {
    node.name
    for node in saved_tree.body
    if isinstance(node, ast.FunctionDef)
}

assert "write_incremental_orders" in function_names
assert "write_incremental_orders_with_retry" in function_names
assert "retry_transient_database_operation" in function_names

print("Retry lifecycle patch: PASS")
print("Saved Python syntax validation: PASS")
print("Required functions verified: PASS")
print(f"Updated file: {source_path}")
print("Neon database accessed: No")
print("Database records modified: No")

Retry lifecycle patch: PASS
Saved Python syntax validation: PASS
Required functions verified: PASS
Updated file: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/incremental.py
Neon database accessed: No
Database records modified: No


In [68]:
import psycopg
from unittest.mock import patch

# Confirm we're testing the current module.
assert hasattr(incremental, "write_incremental_orders_with_retry")

test_records = [
    {
        "order_id": "test_order_001",
        "customer_id": "test_customer_001",
        "order_status": "processing",
        "order_purchase_timestamp": None,
    }
]

# Test 1: transient error followed by success
write_calls = []
failure_calls = []

def transient_then_success(*args, **kwargs):
    write_calls.append(1)
    if len(write_calls) == 1:
        raise psycopg.OperationalError("Simulated temporary connection error")
    return {
        "source_rows": 1,
        "inserted_rows": 1,
        "updated_rows": 0,
    }

with (
    patch.object(
        incremental,
        "write_incremental_orders",
        side_effect=transient_then_success,
    ),
    patch.object(
        incremental,
        "mark_batch_failed",
        side_effect=lambda *args: failure_calls.append(args),
    ),
):
    result = incremental.write_incremental_orders_with_retry(
        "mock://unused",
        "test_retry_success",
        test_records,
        max_attempts=3,
        initial_delay_seconds=0,
        max_delay_seconds=0,
    )

assert len(write_calls) == 2, f"Expected 2 attempts, got {len(write_calls)}"
assert len(failure_calls) == 0, "Successful retry must not mark batch failed"
assert result["inserted_rows"] == 1
print("Test 1 — transient error then success: PASS")


# Test 2: permanent error should not be retried
write_calls = []
failure_calls = []

def permanent_error(*args, **kwargs):
    write_calls.append(1)
    raise ValueError("Simulated permanent validation error")

with (
    patch.object(
        incremental,
        "write_incremental_orders",
        side_effect=permanent_error,
    ),
    patch.object(
        incremental,
        "mark_batch_failed",
        side_effect=lambda *args: failure_calls.append(args),
    ),
):
    try:
        incremental.write_incremental_orders_with_retry(
            "mock://unused",
            "test_permanent_error",
            test_records,
            max_attempts=3,
            initial_delay_seconds=0,
            max_delay_seconds=0,
        )
        raise AssertionError("Expected ValueError was not raised")
    except ValueError as exc:
        assert "permanent validation error" in str(exc)

assert len(write_calls) == 1, f"Expected 1 attempt, got {len(write_calls)}"
assert len(failure_calls) == 1, "Expected one final failure-recording attempt"
print("Test 2 — permanent error is not retried: PASS")


# Test 3: transient errors exhaust the retry limit
write_calls = []
failure_calls = []

def always_transient_error(*args, **kwargs):
    write_calls.append(1)
    raise psycopg.OperationalError("Simulated persistent connection error")

with (
    patch.object(
        incremental,
        "write_incremental_orders",
        side_effect=always_transient_error,
    ),
    patch.object(
        incremental,
        "mark_batch_failed",
        side_effect=lambda *args: failure_calls.append(args),
    ),
):
    try:
        incremental.write_incremental_orders_with_retry(
            "mock://unused",
            "test_retry_exhausted",
            test_records,
            max_attempts=3,
            initial_delay_seconds=0,
            max_delay_seconds=0,
        )
        raise AssertionError("Expected OperationalError was not raised")
    except psycopg.OperationalError as exc:
        assert "persistent connection error" in str(exc)

assert len(write_calls) == 3, f"Expected 3 attempts, got {len(write_calls)}"
assert len(failure_calls) == 1, "Expected one final failure-recording attempt"
print("Test 3 — exhausted retries mark failure once: PASS")

print("\nAll 3 retry lifecycle tests passed.")
print("Database accessed: No")

ERROR:sales_pipeline.incremental:Database operation exhausted retries: attempts=3 error_type=OperationalError


Test 1 — transient error then success: PASS
Test 2 — permanent error is not retried: PASS
Test 3 — exhausted retries mark failure once: PASS

All 3 retry lifecycle tests passed.
Database accessed: No


In [69]:
import ast
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)
SOURCE_FILE = PROJECT_ROOT / "src" / "sales_pipeline" / "incremental.py"

source = SOURCE_FILE.read_text(encoding="utf-8")
tree = ast.parse(source)
lines = source.splitlines()

functions_to_inspect = {
    "register_batch",
    "mark_batch_succeeded",
    "mark_batch_failed",
    "write_incremental_orders",
    "validate_and_quarantine_orders",
    "write_incremental_orders_with_retry",
}

for node in tree.body:
    if isinstance(node, ast.FunctionDef) and node.name in functions_to_inspect:
        print(f"\n{'=' * 20} {node.name} {'=' * 20}")
        for line_number in range(node.lineno, node.end_lineno + 1):
            print(f"{line_number}: {lines[line_number - 1]}")

print("\nInspection complete. No file or database changes made.")


==================== register_batch ====================
34: def register_batch(
35:     database_url: str,
36:     batch_id: str,
37:     batch_type: str = "incremental",
38: ) -> str:
39:     """Register a batch or retry a failed batch.
40: 
41:     Returns 'started' for a new or retried batch and 'skipped' for
42:     an already successful batch. Concurrent running batches are rejected.
43:     """
44:     if not isinstance(batch_id, str) or not batch_id.strip():
45:         raise ValueError("batch_id cannot be empty")
46:     if batch_type not in {"incremental", "full_refresh"}:
47:         raise ValueError("Unsupported batch_type")
48: 
49:     with psycopg.connect(database_url) as conn:
50:         with conn.cursor() as cur:
51:             cur.execute(
52:                 """
53:                 INSERT INTO public.pipeline_batch_runs
54:                     (batch_id, batch_type, status, attempt_number)
55:                 VALUES (%s, %s, 'running', 1)
56:                 ON CO

In [70]:
import ast
import importlib
import re
import sys
from pathlib import Path

# 1. Locate the project file
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)
SOURCE_FILE = PROJECT_ROOT / "src" / "sales_pipeline" / "incremental.py"

if not SOURCE_FILE.exists():
    raise FileNotFoundError(f"Source file not found: {SOURCE_FILE}")

source = SOURCE_FILE.read_text(encoding="utf-8")
original_source = source


def replace_once(text, old, new, description):
    """Replace one exact source fragment or stop without saving."""
    count = text.count(old)
    if count != 1:
        raise RuntimeError(
            f"{description}: expected one matching fragment, found {count}. "
            "No file changes have been saved."
        )
    return text.replace(old, new, 1)


# 2. Update the incremental writer signature to accept original source
# and rejected-record counts while preserving existing defaults.
source = replace_once(
    source,
    '''def write_incremental_orders(
    database_url: str,
    batch_id: str,
    records: list[dict], *, _mark_failed_on_error: bool = True) -> dict[str, int]:''',
    '''def write_incremental_orders(
    database_url: str,
    batch_id: str,
    records: list[dict],
    *,
    source_rows: int | None = None,
    rejected_rows: int = 0,
    _mark_failed_on_error: bool = True,
) -> dict[str, int]:''',
    "Writer signature",
)

# 3. Validate the supplied counts and preserve the default behavior.
source = replace_once(
    source,
    '''    try:
        required_fields = {
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
        }

        if not isinstance(records, list):
            raise TypeError("records must be a list of dictionaries")

        seen_ids = set()''',
    '''    try:
        if source_rows is not None and (
            not isinstance(source_rows, int)
            or isinstance(source_rows, bool)
            or source_rows < 0
        ):
            raise ValueError("source_rows must be a non-negative integer")

        if (
            not isinstance(rejected_rows, int)
            or isinstance(rejected_rows, bool)
            or rejected_rows < 0
        ):
            raise ValueError("rejected_rows must be a non-negative integer")

        required_fields = {
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
        }

        if not isinstance(records, list):
            raise TypeError("records must be a list of dictionaries")

        source_rows_effective = (
            len(records) if source_rows is None else source_rows
        )

        seen_ids = set()''',
    "Writer count validation",
)

# 4. Store the real source and rejection counts in batch metadata.
source = replace_once(
    source,
    '''                    SET status = 'succeeded',
                        finished_at = NOW(),
                        source_rows = %s,
                        inserted_rows = %s,
                        updated_rows = %s,
                        rejected_rows = 0,
                        error_message = NULL''',
    '''                    SET status = 'succeeded',
                        finished_at = NOW(),
                        source_rows = %s,
                        inserted_rows = %s,
                        updated_rows = %s,
                        rejected_rows = %s,
                        error_message = NULL''',
    "Batch metadata SQL",
)

source = replace_once(
    source,
    '''                    (
                        len(records),
                        inserted_rows,
                        updated_rows,
                        batch_id,
                    ),''',
    '''                    (
                        source_rows_effective,
                        inserted_rows,
                        updated_rows,
                        rejected_rows,
                        batch_id,
                    ),''',
    "Batch metadata parameters",
)

# 5. Update logging and returned counts.
source = replace_once(
    source,
    '''        logger.info(
            "Incremental write committed: batch_id=%s "
            "source_rows=%s inserted_rows=%s updated_rows=%s",
            batch_id, len(records), inserted_rows, updated_rows,
        )
        return {
            "source_rows": len(records),
            "inserted_rows": inserted_rows,
            "updated_rows": updated_rows,
            "rejected_rows": 0,
        }''',
    '''        logger.info(
            "Incremental write committed: batch_id=%s "
            "source_rows=%s inserted_rows=%s updated_rows=%s "
            "rejected_rows=%s",
            batch_id,
            source_rows_effective,
            inserted_rows,
            updated_rows,
            rejected_rows,
        )
        return {
            "source_rows": source_rows_effective,
            "inserted_rows": inserted_rows,
            "updated_rows": updated_rows,
            "rejected_rows": rejected_rows,
        }''',
    "Writer logging and result",
)

# 6. Extend the retry wrapper with the same optional counts.
source = replace_once(
    source,
    '''def write_incremental_orders_with_retry(
    database_url: str,
    batch_id: str,
    records: list[dict],
    *,
    max_attempts: int = 3,''',
    '''def write_incremental_orders_with_retry(
    database_url: str,
    batch_id: str,
    records: list[dict],
    *,
    source_rows: int | None = None,
    rejected_rows: int = 0,
    max_attempts: int = 3,''',
    "Retry wrapper signature",
)

source = replace_once(
    source,
    '''                batch_id,
                records,
                _mark_failed_on_error=False,''',
    '''                batch_id,
                records,
                source_rows=source_rows,
                rejected_rows=rejected_rows,
                _mark_failed_on_error=False,''',
    "Retry wrapper writer call",
)

# 7. Add an orchestration entry point.
tree = ast.parse(source)
existing_functions = {
    node.name
    for node in tree.body
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
}

if "process_incremental_orders" in existing_functions:
    raise RuntimeError(
        "process_incremental_orders already exists. "
        "No file changes have been saved."
    )

orchestrator = '''

def process_incremental_orders(
    database_url: str,
    batch_id: str,
    records: list[dict],
    *,
    max_attempts: int = 3,
    initial_delay_seconds: float = 1.0,
    max_delay_seconds: float = 8.0,
) -> dict:
    """Register, validate, quarantine, and incrementally write an order batch.

    A previously successful batch is skipped. Invalid records are quarantined,
    while valid records are written transactionally with bounded retries.
    """
    registration = register_batch(
        database_url,
        batch_id,
        batch_type="incremental",
    )

    if registration == "skipped":
        return {
            "status": "skipped",
            "batch_id": batch_id,
            "source_rows": 0,
            "inserted_rows": 0,
            "updated_rows": 0,
            "rejected_rows": 0,
        }

    try:
        validation = validate_and_quarantine_orders(
            database_url,
            batch_id,
            records,
        )
    except Exception as exc:
        try:
            mark_batch_failed(
                database_url,
                batch_id,
                f"{type(exc).__name__}: {exc}",
            )
        except Exception:
            logger.exception(
                "Could not mark batch %s as failed after validation "
                "or quarantine error",
                batch_id,
            )
        raise

    result = write_incremental_orders_with_retry(
        database_url,
        batch_id,
        validation["valid_records"],
        source_rows=validation["source_rows"],
        rejected_rows=validation["rejected_rows"],
        max_attempts=max_attempts,
        initial_delay_seconds=initial_delay_seconds,
        max_delay_seconds=max_delay_seconds,
    )

    result["status"] = "succeeded"
    result["batch_id"] = batch_id
    return result
'''

source = source.rstrip() + "\n\n" + orchestrator.lstrip()

# 8. Validate everything before saving.
ast.parse(source)

required_functions = {
    "write_incremental_orders",
    "write_incremental_orders_with_retry",
    "validate_and_quarantine_orders",
    "register_batch",
    "mark_batch_failed",
    "process_incremental_orders",
}

updated_tree = ast.parse(source)
updated_functions = {
    node.name
    for node in updated_tree.body
    if isinstance(node, ast.FunctionDef)
}

missing = required_functions - updated_functions
if missing:
    raise RuntimeError(
        f"Missing expected functions: {sorted(missing)}. No file changes saved."
    )

# Save only after all checks pass.
SOURCE_FILE.write_text(source, encoding="utf-8")

# 9. Verify the saved source and reload the module.
saved_source = SOURCE_FILE.read_text(encoding="utf-8")
ast.parse(saved_source)

source_dir = str(PROJECT_ROOT / "src")
if source_dir in sys.path:
    sys.path.remove(source_dir)
sys.path.insert(0, source_dir)

import sales_pipeline.incremental as incremental
incremental = importlib.reload(incremental)

for function_name in required_functions:
    assert hasattr(incremental, function_name), (
        f"Function missing after reload: {function_name}"
    )

print("Incremental writer count handling: PASS")
print("Retry wrapper count forwarding: PASS")
print("Orchestration function added: PASS")
print("Saved Python syntax validation: PASS")
print("Module reload and function checks: PASS")
print(f"Updated file: {SOURCE_FILE}")
print("Neon database accessed: No")
print("Database records modified: No")

Incremental writer count handling: PASS
Retry wrapper count forwarding: PASS
Orchestration function added: PASS
Saved Python syntax validation: PASS
Module reload and function checks: PASS
Updated file: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/incremental.py
Neon database accessed: No
Database records modified: No


In [71]:
import uuid
import psycopg
from google.colab import userdata

# 1. Get the database URL without printing credentials.
database_url = userdata.get("NEON_DB_URL")
if not database_url:
    raise RuntimeError("NEON_DB_URL is missing from Colab Secrets.")

# 2. Create unique synthetic test identifiers.
test_token = uuid.uuid4().hex[:12]
batch_id = f"phase5_orchestration_test_{test_token}"
valid_order_id = f"phase5_valid_{test_token}"
valid_customer_id = f"phase5_customer_{test_token}"

records = [
    {
        "order_id": valid_order_id,
        "customer_id": valid_customer_id,
        "order_status": "processing",
        "order_purchase_timestamp": None,
    },
    {
        # Invalid: missing customer_id and other required fields.
        "order_id": f"phase5_missing_fields_{test_token}",
    },
    {
        # Invalid: blank order_id.
        "order_id": " ",
        "customer_id": f"phase5_customer_invalid_{test_token}",
        "order_status": "processing",
        "order_purchase_timestamp": None,
    },
    {
        # Invalid: duplicate of the first record's order_id.
        "order_id": valid_order_id,
        "customer_id": f"phase5_duplicate_customer_{test_token}",
        "order_status": "shipped",
        "order_purchase_timestamp": None,
    },
]

# 3. Run the full incremental orchestration.
first_result = incremental.process_incremental_orders(
    database_url,
    batch_id,
    records,
    max_attempts=3,
    initial_delay_seconds=0,
    max_delay_seconds=0,
)

assert first_result["status"] == "succeeded", first_result
assert first_result["source_rows"] == 4, first_result
assert first_result["inserted_rows"] == 1, first_result
assert first_result["updated_rows"] == 0, first_result
assert first_result["rejected_rows"] == 3, first_result

print("Test 1 — mixed valid and invalid batch: PASS")
print("  Source rows:", first_result["source_rows"])
print("  Inserted rows:", first_result["inserted_rows"])
print("  Updated rows:", first_result["updated_rows"])
print("  Rejected rows:", first_result["rejected_rows"])

# 4. Verify persisted batch metadata, quarantine rows, and target record.
with psycopg.connect(database_url) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT status, source_rows, inserted_rows,
                   updated_rows, rejected_rows, attempt_number
            FROM public.pipeline_batch_runs
            WHERE batch_id = %s
            """,
            (batch_id,),
        )
        batch_row = cur.fetchone()
        assert batch_row is not None, "Batch metadata row not found."

        cur.execute(
            """
            SELECT COUNT(*)
            FROM public.phase5_rejected_records
            WHERE batch_id = %s
            """,
            (batch_id,),
        )
        quarantine_count = cur.fetchone()[0]

        cur.execute(
            """
            SELECT COUNT(*)
            FROM public.phase5_incremental_orders
            WHERE order_id = %s AND source_batch_id = %s
            """,
            (valid_order_id, batch_id),
        )
        target_count = cur.fetchone()[0]

assert batch_row[0] == "succeeded", batch_row
assert batch_row[1:5] == (4, 1, 0, 3), batch_row
assert quarantine_count == 3, quarantine_count
assert target_count == 1, target_count

print("Test 2 — Neon batch metadata and quarantine: PASS")
print("  Persisted batch metadata:", batch_row)
print("  Quarantined records:", quarantine_count)
print("  Matching target records:", target_count)

# 5. Rerun the exact same batch; it should be skipped.
second_result = incremental.process_incremental_orders(
    database_url,
    batch_id,
    records,
    max_attempts=3,
    initial_delay_seconds=0,
    max_delay_seconds=0,
)

assert second_result["status"] == "skipped", second_result

with psycopg.connect(database_url) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT COUNT(*)
            FROM public.phase5_incremental_orders
            WHERE order_id = %s AND source_batch_id = %s
            """,
            (valid_order_id, batch_id),
        )
        target_count_after_rerun = cur.fetchone()[0]

        cur.execute(
            """
            SELECT COUNT(*)
            FROM public.phase5_rejected_records
            WHERE batch_id = %s
            """,
            (batch_id,),
        )
        quarantine_count_after_rerun = cur.fetchone()[0]

assert target_count_after_rerun == 1
assert quarantine_count_after_rerun == 3

print("Test 3 — successful batch rerun is skipped: PASS")
print("  Target records after rerun:", target_count_after_rerun)
print("  Quarantine records after rerun:", quarantine_count_after_rerun)
print("\nAll Phase 5 orchestration integration checks passed.")
print("Only isolated Phase 5 test tables and batch metadata were used.")

INFO:sales_pipeline.incremental:Batch registered: batch_id=phase5_orchestration_test_38e197a55725 type=incremental attempt=1
INFO:sales_pipeline.incremental:Order validation completed: batch_id=phase5_orchestration_test_38e197a55725 source_rows=4 valid_rows=1 rejected_rows=3
INFO:sales_pipeline.incremental:Incremental write committed: batch_id=phase5_orchestration_test_38e197a55725 source_rows=4 inserted_rows=1 updated_rows=0 rejected_rows=3


Test 1 — mixed valid and invalid batch: PASS
  Source rows: 4
  Inserted rows: 1
  Updated rows: 0
  Rejected rows: 3
Test 2 — Neon batch metadata and quarantine: PASS
  Persisted batch metadata: ('succeeded', 4, 1, 0, 3, 1)
  Quarantined records: 3
  Matching target records: 1


INFO:sales_pipeline.incremental:Batch skipped because it already succeeded: batch_id=phase5_orchestration_test_38e197a55725


Test 3 — successful batch rerun is skipped: PASS
  Target records after rerun: 1
  Quarantine records after rerun: 3

All Phase 5 orchestration integration checks passed.
Only isolated Phase 5 test tables and batch metadata were used.


In [72]:
import os
import sys
import subprocess
from pathlib import Path
from google.colab import userdata

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

database_url = userdata.get("NEON_DB_URL")
if not database_url:
    raise RuntimeError("NEON_DB_URL is missing from Colab Secrets.")

env = os.environ.copy()
env["NEON_DB_URL"] = database_url
env["PYTHONPATH"] = (
    str(PROJECT_ROOT / "src")
    + os.pathsep
    + str(PROJECT_ROOT)
    + os.pathsep
    + env.get("PYTHONPATH", "")
)

test_modules = [
    "tests.test_incremental",
    "tests.test_incremental_integration",
    "tests.test_pipeline",
    "tests.test_phase4_storage_modeling",
    "tests.test_spark_transformations",
]

command = [
    sys.executable,
    "-m",
    "unittest",
    *test_modules,
    "-v",
]

print("Running consolidated regression suite...")
print("Test modules:", len(test_modules))
print("Database credentials: hidden")
print("Git operations: none\n")

result = subprocess.run(
    command,
    cwd=str(PROJECT_ROOT),
    env=env,
    text=True,
    check=False,
)

if result.returncode != 0:
    raise RuntimeError(
        f"Regression suite failed with exit code {result.returncode}. "
        "Review the test output above before proceeding."
    )

print("\nConsolidated regression suite: PASS")

Running consolidated regression suite...
Test modules: 5
Database credentials: hidden
Git operations: none


Consolidated regression suite: PASS


reruns the same five modules and displays the test runner's output, including stderr

In [73]:
import os
import sys
import subprocess
from pathlib import Path
from google.colab import userdata

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

database_url = userdata.get("NEON_DB_URL")
if not database_url:
    raise RuntimeError("NEON_DB_URL is missing from Colab Secrets.")

env = os.environ.copy()
env["NEON_DB_URL"] = database_url
env["PYTHONPATH"] = os.pathsep.join([
    str(PROJECT_ROOT / "src"),
    str(PROJECT_ROOT),
    env.get("PYTHONPATH", ""),
])

command = [
    sys.executable,
    "-m",
    "unittest",
    "tests.test_incremental",
    "tests.test_incremental_integration",
    "tests.test_pipeline",
    "tests.test_phase4_storage_modeling",
    "tests.test_spark_transformations",
    "-v",
]

result = subprocess.run(
    command,
    cwd=str(PROJECT_ROOT),
    env=env,
    text=True,
    capture_output=True,
    check=False,
)

print("=== Test output ===")
print(result.stdout)
print(result.stderr)

print("\n=== Summary ===")
print("Exit code:", result.returncode)
print("Result:", "PASS" if result.returncode == 0 else "FAIL")
print("Git operations: none")

if result.returncode != 0:
    raise RuntimeError("Regression suite failed. Review the output above.")

=== Test output ===
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SALES ANALYTICS DATA PIPELINE

[1/5] Validating source data...

[2/5] Building order analytics...

[3/5] Building product and category analytics...

[4/5] Building customer analytics...

[5/5] Building sales performance analytics...

PIPELINE COMPLETED SUCCESSFULLY
Duration: 0.0 seconds
Validation checks: 1
Order rows: 2

In [74]:
import ast
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

files_to_inspect = [
    PROJECT_ROOT / "src" / "sales_pipeline" / "incremental.py",
    PROJECT_ROOT / "tests" / "test_incremental.py",
]

for file_path in files_to_inspect:
    print(f"\n{'=' * 16} {file_path.name} {'=' * 16}")
    source = file_path.read_text(encoding="utf-8")
    lines = source.splitlines()

    if file_path.name == "incremental.py":
        tree = ast.parse(source)
        for node in tree.body:
            if (
                isinstance(node, ast.FunctionDef)
                and node.name in {
                    "validate_and_quarantine_orders",
                    "write_incremental_orders",
                }
            ):
                print(f"\n--- {node.name} (lines {node.lineno}-{node.end_lineno}) ---")
                for i in range(node.lineno, node.end_lineno + 1):
                    print(f"{i}: {lines[i - 1]}")
    else:
        for i, line in enumerate(lines, start=1):
            print(f"{i}: {line}")

print("\nInspection complete. No files or database records changed.")


================ incremental.py ================

--- write_incremental_orders (lines 208-413) ---
208: def write_incremental_orders(
209:     database_url: str,
210:     batch_id: str,
211:     records: list[dict],
212:     *,
213:     source_rows: int | None = None,
214:     rejected_rows: int = 0,
215:     _mark_failed_on_error: bool = True,
216: ) -> dict[str, int]:
217:     """Atomically upsert orders and record batch success.
218: 
219:     If validation or database processing fails, attempt to mark
220:     the registered batch as failed after the write transaction rolls back.
221:     """
222:     try:
223:         if source_rows is not None and (
224:             not isinstance(source_rows, int)
225:             or isinstance(source_rows, bool)
226:             or source_rows < 0
227:         ):
228:             raise ValueError("source_rows must be a non-negative integer")
229: 
230:         if (
231:             not isinstance(rejected_rows, int)
232:             or isinsta

In [75]:
import ast
import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)
SOURCE_FILE = PROJECT_ROOT / "src" / "sales_pipeline" / "incremental.py"

if not SOURCE_FILE.exists():
    raise FileNotFoundError(f"Source file not found: {SOURCE_FILE}")

source = SOURCE_FILE.read_text(encoding="utf-8")
original_source = source


def replace_once(text, old, new, description):
    count = text.count(old)
    if count != 1:
        raise RuntimeError(
            f"{description}: expected exactly one match, found {count}. "
            "No changes have been saved."
        )
    return text.replace(old, new, 1)


# 1. Add a shared validator for order_status and order_purchase_timestamp.
helper = '''def _validate_order_fields(record: dict) -> str | None:
    """Return a validation error for order status or purchase timestamp."""
    import datetime

    status = record["order_status"]
    if not isinstance(status, str) or not status.strip():
        return "Invalid order_status: expected a non-empty string"

    timestamp = record["order_purchase_timestamp"]

    # Missing timestamps remain permitted by the current pipeline contract.
    if timestamp is None:
        return None

    if isinstance(timestamp, datetime.datetime):
        return None

    if isinstance(timestamp, str):
        value = timestamp.strip()
        if not value:
            return "Invalid order_purchase_timestamp: blank timestamp"

        try:
            datetime.datetime.fromisoformat(
                value.replace("Z", "+00:00")
            )
        except ValueError:
            return (
                "Invalid order_purchase_timestamp: "
                "expected an ISO-formatted timestamp"
            )
        return None

    return (
        "Invalid order_purchase_timestamp: expected datetime, "
        "ISO-formatted string, or None"
    )


'''

if "def _validate_order_fields(" in source:
    raise RuntimeError(
        "Shared validator already exists. No changes have been saved."
    )

source = replace_once(
    source,
    "def write_incremental_orders(\n",
    helper + "def write_incremental_orders(\n",
    "Shared validator insertion",
)

# 2. Enforce these rules in the direct database writer.
source = replace_once(
    source,
    '''            if order_id in seen_ids:
                raise ValueError(
                    f"Duplicate order_id within input batch: {order_id}"
                )''',
    '''            field_error = _validate_order_fields(record)
            if field_error:
                raise ValueError(f"Record {index}: {field_error}")

            if order_id in seen_ids:
                raise ValueError(
                    f"Duplicate order_id within input batch: {order_id}"
                )''',
    "Direct writer field validation",
)

# 3. Apply the same rules during quarantine validation.
source = replace_once(
    source,
    '''            elif record["order_id"] in seen_ids:
                reason = f"Duplicate order_id within batch: {record['order_id']}"''',
    '''            elif (field_error := _validate_order_fields(record)) is not None:
                reason = field_error
            elif record["order_id"] in seen_ids:
                reason = f"Duplicate order_id within batch: {record['order_id']}"''',
    "Quarantine field validation",
)

# 4. Validate the full source before saving.
ast.parse(source)

tree = ast.parse(source)
function_names = {
    node.name for node in tree.body
    if isinstance(node, ast.FunctionDef)
}

required_functions = {
    "_validate_order_fields",
    "write_incremental_orders",
    "validate_and_quarantine_orders",
    "write_incremental_orders_with_retry",
    "process_incremental_orders",
}

missing = required_functions - function_names
if missing:
    raise RuntimeError(
        f"Expected functions missing: {sorted(missing)}. "
        "No changes have been saved."
    )

# 5. Save and verify the saved file.
SOURCE_FILE.write_text(source, encoding="utf-8")
saved_source = SOURCE_FILE.read_text(encoding="utf-8")
ast.parse(saved_source)

# 6. Reload the current module.
source_dir = str(PROJECT_ROOT / "src")
if source_dir in sys.path:
    sys.path.remove(source_dir)
sys.path.insert(0, source_dir)

import sales_pipeline.incremental as incremental
incremental = importlib.reload(incremental)

assert hasattr(incremental, "_validate_order_fields")
assert hasattr(incremental, "process_incremental_orders")

# 7. Run focused, database-free checks on the shared validator.
from datetime import datetime

valid_record = {
    "order_id": "test_order",
    "customer_id": "test_customer",
    "order_status": "processing",
    "order_purchase_timestamp": None,
}

assert incremental._validate_order_fields(valid_record) is None

valid_iso_record = {
    **valid_record,
    "order_purchase_timestamp": "2018-04-01 00:27:51",
}
assert incremental._validate_order_fields(valid_iso_record) is None

valid_datetime_record = {
    **valid_record,
    "order_purchase_timestamp": datetime(2018, 4, 1, 0, 27, 51),
}
assert incremental._validate_order_fields(valid_datetime_record) is None

invalid_status_record = {
    **valid_record,
    "order_status": " ",
}
assert "order_status" in incremental._validate_order_fields(
    invalid_status_record
)

invalid_timestamp_record = {
    **valid_record,
    "order_purchase_timestamp": "not-a-timestamp",
}
assert "timestamp" in incremental._validate_order_fields(
    invalid_timestamp_record
)

print("Shared field validator added: PASS")
print("Direct writer validation updated: PASS")
print("Quarantine validation updated: PASS")
print("ISO timestamp, datetime, and None checks: PASS")
print("Invalid status and timestamp checks: PASS")
print("Saved Python syntax and module reload: PASS")
print("Neon database accessed: No")
print("Database records modified: No")

Shared field validator added: PASS
Direct writer validation updated: PASS
Quarantine validation updated: PASS
ISO timestamp, datetime, and None checks: PASS
Invalid status and timestamp checks: PASS
Saved Python syntax and module reload: PASS
Neon database accessed: No
Database records modified: No


In [76]:
import ast
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

TEST_FILE = PROJECT_ROOT / "tests" / "test_incremental.py"

if not TEST_FILE.exists():
    raise FileNotFoundError(f"Test file not found: {TEST_FILE}")

source = TEST_FILE.read_text(encoding="utf-8")

test_class_name = "TestIncrementalOrderFieldValidation"

if f"class {test_class_name}" in source:
    raise RuntimeError(
        "The new validation test class already exists. "
        "No file changes have been made."
    )

new_tests = '''

class TestIncrementalOrderFieldValidation(unittest.TestCase):
    """Tests for order status and purchase timestamp validation."""

    def setUp(self):
        from unittest.mock import patch

        self.connect_patcher = patch(
            "sales_pipeline.incremental.psycopg.connect",
            side_effect=AssertionError(
                "Database connection should not be attempted"
            ),
        )
        self.connect_patcher.start()
        self.addCleanup(self.connect_patcher.stop)

    def setUp_record(self):
        return {
            "order_id": "order_validation_test",
            "customer_id": "customer_validation_test",
            "order_status": "processing",
            "order_purchase_timestamp": None,
        }

    def test_none_timestamp_is_valid(self):
        from sales_pipeline.incremental import _validate_order_fields

        record = self.setUp_record()
        self.assertIsNone(_validate_order_fields(record))

    def test_iso_timestamp_string_is_valid(self):
        from sales_pipeline.incremental import _validate_order_fields

        record = self.setUp_record()
        record["order_purchase_timestamp"] = "2018-04-01 00:27:51"
        self.assertIsNone(_validate_order_fields(record))

    def test_datetime_object_is_valid(self):
        from datetime import datetime
        from sales_pipeline.incremental import _validate_order_fields

        record = self.setUp_record()
        record["order_purchase_timestamp"] = datetime(2018, 4, 1, 0, 27, 51)
        self.assertIsNone(_validate_order_fields(record))

    def test_invalid_timestamp_string_is_rejected(self):
        from sales_pipeline.incremental import _validate_order_fields

        record = self.setUp_record()
        record["order_purchase_timestamp"] = "not-a-timestamp"

        error = _validate_order_fields(record)
        self.assertIsNotNone(error)
        self.assertIn("timestamp", error)

    def test_blank_timestamp_string_is_rejected(self):
        from sales_pipeline.incremental import _validate_order_fields

        record = self.setUp_record()
        record["order_purchase_timestamp"] = "   "

        error = _validate_order_fields(record)
        self.assertIsNotNone(error)
        self.assertIn("timestamp", error)

    def test_blank_order_status_is_rejected(self):
        from sales_pipeline.incremental import _validate_order_fields

        record = self.setUp_record()
        record["order_status"] = " "

        error = _validate_order_fields(record)
        self.assertIsNotNone(error)
        self.assertIn("order_status", error)

    def test_non_string_order_status_is_rejected(self):
        from sales_pipeline.incremental import _validate_order_fields

        record = self.setUp_record()
        record["order_status"] = None

        error = _validate_order_fields(record)
        self.assertIsNotNone(error)
        self.assertIn("order_status", error)

    def test_direct_writer_rejects_invalid_timestamp_before_connect(self):
        from sales_pipeline.incremental import write_incremental_orders

        record = self.setUp_record()
        record["order_purchase_timestamp"] = "invalid-date"

        with self.assertRaisesRegex(ValueError, "timestamp"):
            write_incremental_orders("unused", "batch_validation_test", [record])

    def test_direct_writer_rejects_blank_status_before_connect(self):
        from sales_pipeline.incremental import write_incremental_orders

        record = self.setUp_record()
        record["order_status"] = " "

        with self.assertRaisesRegex(ValueError, "order_status"):
            write_incremental_orders("unused", "batch_validation_test", [record])
'''

updated_source = source.rstrip() + "\n\n" + new_tests.lstrip()

# Validate before saving.
ast.parse(updated_source)
TEST_FILE.write_text(updated_source, encoding="utf-8")
ast.parse(TEST_FILE.read_text(encoding="utf-8"))

print("Permanent validation tests added: PASS")
print("Test file syntax validation: PASS")

# Run incremental unit tests without supplying database credentials.
env = __import__("os").environ.copy()
env["PYTHONPATH"] = (
    str(PROJECT_ROOT / "src")
    + __import__("os").pathsep
    + str(PROJECT_ROOT)
    + __import__("os").pathsep
    + env.get("PYTHONPATH", "")
)
env.pop("NEON_DB_URL", None)

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "tests.test_incremental",
        "-v",
    ],
    cwd=str(PROJECT_ROOT),
    env=env,
    text=True,
    check=False,
)

if result.returncode != 0:
    raise RuntimeError(
        "Incremental unit tests failed. Review the test output above."
    )

print("\nIncremental unit test suite: PASS")
print("Database accessed: No")

Permanent validation tests added: PASS
Test file syntax validation: PASS

Incremental unit test suite: PASS
Database accessed: No


confirming that valid timestamps are accepted, malformed timestamps and blank statuses are rejected, and direct writes fail validation

In [77]:
import os
import re
import sys
import subprocess
from pathlib import Path
from google.colab import userdata

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

database_url = userdata.get("NEON_DB_URL")
if not database_url:
    raise RuntimeError("NEON_DB_URL is missing from Colab Secrets.")

env = os.environ.copy()
env["NEON_DB_URL"] = database_url
env["PYTHONPATH"] = os.pathsep.join([
    str(PROJECT_ROOT / "src"),
    str(PROJECT_ROOT),
    env.get("PYTHONPATH", ""),
])

test_modules = [
    "tests.test_incremental",
    "tests.test_incremental_integration",
    "tests.test_pipeline",
    "tests.test_phase4_storage_modeling",
    "tests.test_spark_transformations",
]

result = subprocess.run(
    [sys.executable, "-m", "unittest", *test_modules, "-v"],
    cwd=str(PROJECT_ROOT),
    env=env,
    text=True,
    capture_output=True,
    check=False,
)

# unittest normally writes its verbose output to stderr.
output = result.stdout + "\n" + result.stderr
print(output)

summary = re.findall(
    r"Ran (\d+) tests? in ([\d.]+)s",
    output,
)

print("\n=== Consolidated regression summary ===")
if summary:
    count, elapsed = summary[-1]
    print(f"Tests run: {count}")
    print(f"Duration: {elapsed} seconds")
else:
    print("Test count could not be extracted from the runner output.")

print("Exit code:", result.returncode)
print("Result:", "PASS" if result.returncode == 0 else "FAIL")
print("Git operations: none")

if result.returncode != 0:
    raise RuntimeError(
        "Regression suite failed. Review the individual test output above."
    )

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SALES ANALYTICS DATA PIPELINE

[1/5] Validating source data...

[2/5] Building order analytics...

[3/5] Building product and category analytics...

[4/5] Building customer analytics...

[5/5] Building sales performance analytics...

PIPELINE COMPLETED SUCCESSFULLY
Duration: 0.0 seconds
Validation checks: 1
Order rows: 2
Product rows: 1
Cat

### Reliability checks

Audit

In [78]:
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

checks = [
    ("Git status", ["git", "status", "--short"]),
    ("Diff summary", ["git", "diff", "--stat"]),
    ("Whitespace validation", ["git", "diff", "--check"]),
    (
        "Incremental retry implementation",
        ["git", "diff", "--", "src/sales_pipeline/incremental.py"],
    ),
]

for title, command in checks:
    print(f"\n=== {title} ===")
    result = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
        check=False,
    )
    print(result.stdout or "(no output)")
    if result.stderr:
        print("STDERR:", result.stderr)
    print("Exit code:", result.returncode)


=== Git status ===
 M 02_postgresql_data_warehouse.ipynb
 M 03_PySpark_transformation.ipynb
 M 04_Storage_and_Data_modeling.ipynb
?? 05_Incremental_processing_and_Reliability.ipynb
?? src/sales_pipeline/incremental.py
?? tests/test_incremental.py
?? tests/test_incremental_integration.py

Exit code: 0

=== Diff summary ===
 02_postgresql_data_warehouse.ipynb | 2 +-
 03_PySpark_transformation.ipynb    | 2 +-
 04_Storage_and_Data_modeling.ipynb | 2 +-
 3 files changed, 3 insertions(+), 3 deletions(-)

Exit code: 0

=== Whitespace validation ===
(no output)
Exit code: 0

=== Incremental retry implementation ===
(no output)
Exit code: 0


In [79]:
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

files_to_restore = [
    "02_postgresql_data_warehouse.ipynb",
    "03_PySpark_transformation.ipynb",
    "04_Storage_and_Data_modeling.ipynb",
]

result = subprocess.run(
    ["git", "restore", "--"] + files_to_restore,
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
    check=False,
)

if result.returncode != 0:
    print("Restore failed:")
    print(result.stderr)
    raise RuntimeError("Git restore failed.")

print("Restored the three notebooks to their committed versions.")
print("No commit or push performed.")

status = subprocess.run(
    ["git", "status", "--short"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
    check=True,
)

print("\n=== Git status after restoration ===")
print(status.stdout or "Working tree clean.")

Restored the three notebooks to their committed versions.
No commit or push performed.

=== Git status after restoration ===
?? 05_Incremental_processing_and_Reliability.ipynb
?? src/sales_pipeline/incremental.py
?? tests/test_incremental.py
?? tests/test_incremental_integration.py



In [80]:
import os
import subprocess
import tempfile
from pathlib import Path
from google.colab import userdata

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

FILES_TO_PUSH = [
    "05_Incremental_processing_and_Reliability.ipynb",
    "src/sales_pipeline/incremental.py",
    "tests/test_incremental.py",
    "tests/test_incremental_integration.py",
]

EXPECTED_REMOTE = (
    "https://github.com/vipulkumawat17/"
    "Sales-analytics-data-pipeline.git"
)

def run_git(args, env=None):
    result = subprocess.run(
        ["git", *args],
        cwd=PROJECT_ROOT,
        env=env,
        text=True,
        capture_output=True,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Git command failed: git {' '.join(args)}\n"
            f"{result.stderr or result.stdout}"
        )
    return result.stdout.strip()

# 1. Verify repository and branch.
if run_git(["branch", "--show-current"]) != "main":
    raise RuntimeError("Expected branch 'main'. No changes were pushed.")

remote = run_git(["remote", "get-url", "origin"]).rstrip("/")
if remote != EXPECTED_REMOTE.rstrip("/"):
    raise RuntimeError(
        f"Unexpected origin remote: {remote}. No changes were pushed."
    )

# 2. Verify all intended files exist.
for filename in FILES_TO_PUSH:
    if not (PROJECT_ROOT / filename).is_file():
        raise FileNotFoundError(f"Missing Phase 5 file: {filename}")

# 3. Retrieve the token without displaying it.
token = userdata.get("GITHUB_TOKEN")
if not token or not token.strip():
    raise RuntimeError(
        "GITHUB_TOKEN is missing. Add it under Colab Secrets "
        "and enable notebook access."
    )

# 4. Stage only the four Phase 5 files.
run_git(["add", "--", *FILES_TO_PUSH])

staged = run_git(["diff", "--cached", "--name-only"]).splitlines()
if sorted(staged) != sorted(FILES_TO_PUSH):
    raise RuntimeError(
        "Staged files do not match the approved Phase 5 file list. "
        f"Staged: {staged}"
    )

run_git(["diff", "--cached", "--check"])

# 5. Commit the Phase 5 implementation.
run_git([
    "commit",
    "-m",
    "Implement Phase 5 incremental processing and reliability",
])

# 6. Authenticate the push without embedding the token in the remote URL.
with tempfile.TemporaryDirectory() as temp_dir:
    askpass = Path(temp_dir) / "git-askpass.sh"
    askpass.write_text(
        "#!/bin/sh\n"
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *Password*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        '  *) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        "esac\n"
    )
    askpass.chmod(0o700)

    env = os.environ.copy()
    env["GITHUB_TOKEN"] = token
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"

    # Do not print the token or the environment.
    del token

    push_result = subprocess.run(
        ["git", "push", "origin", "main"],
        cwd=PROJECT_ROOT,
        env=env,
        text=True,
        capture_output=True,
        check=False,
    )

    if push_result.returncode != 0:
        raise RuntimeError(
            "Push failed. Check the token's repository permissions "
            "and GitHub authentication.\n"
            f"{push_result.stderr or push_result.stdout}"
        )

    print(push_result.stdout or "Git push completed successfully.")

# 7. Confirm the working tree is clean.
print("\n=== Final Git status ===")
print(run_git(["status", "--short"]) or "Working tree clean.")
print("\nPhase 5 commit and push completed.")

RuntimeError: Unexpected origin remote: https://github.com/vipulkumawat17/Sales-analytics-data-pipeline. No changes were pushed.